Creates combined LULUCF tables (long and wide format) and creates figures
Table manipulation from Claude session 'LULUCF and component table creation'

In [1]:
import pandas as pd
import yaml
import geodatasets
import geopandas as gpd
import math
import rasterio
import os
import openpyxl
import pygwalker as pyg
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.lines as mlines
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import re
import glob
import sys
import textwrap
from shapely.geometry import Point
from datetime import datetime
from pathlib import Path
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from matplotlib.transforms import blended_transform_factory
from matplotlib.gridspec import GridSpec
from io import BytesIO
from IPython.display import Image, display as ipy_display
from rasterio.windows import Window
from tqdm import tqdm

In [2]:
# To add project files
# Keeps going up project structure until it gets to the root.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69a5ff07-91e0-8329-88d1-cf2ea6c159c2
project_root = Path().resolve()
while project_root.name != "AFOLU_GHG_flux_model":
    project_root = project_root.parent

sys.path.append(str(project_root))

from src.utilities import constants_and_names as cn
from src.utilities import universal_utilities as uu

now = datetime.now()
today = now.strftime("%Y%m%d")

pd.set_option('display.max_columns', None)

In [3]:
# Fills in the intervening years of the multi-year intervals for soil data.
# Basically, duplicates values from end-of-interval years for all the preceding years in the interval,
# but only for that row's own nominal interval. Missing intervals stay blank.
# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69c2fe30-7004-8328-9f6d-1c231ba04e20
def fill_in_soil_years(df, group_cols, agg_first=False):

    # Aggregate first
    if agg_first:
        df = (
            df
            .groupby(group_cols, dropna=False, as_index=False)
            .agg({
                "zonal_stats_sum": "sum",
                "area_ha": "sum"
            })
        )

    # Sort by timeseries + year
    df = df.sort_values(series_cols + ["year"]).reset_index(drop=True)

    year_vals = df["year"].astype(int)

    # Maximum (final) year in the dataset
    last_year = year_vals.max()

    # Nominal interval length for each endpoint year:
    # - years divisible by 5 -> 5-year interval
    # - final year in dataset -> interval from 2021 to last_year
    # - everything else defaults to 1 year unless you add more rules
    interval_len = np.where(
        year_vals.eq(last_year),
        last_year - 2021 + 1,
        np.where(year_vals.mod(5).eq(0), 5, 1)
    )

    # Start year comes only from the row's own nominal interval,
    # not from the previous observed year.
    start_year = (year_vals - interval_len + 1).astype(int).to_numpy()
    end_year = year_vals.to_numpy()

    # Number of repetitions for each interval
    n_rep = end_year - start_year + 1

    # Duplicate rows
    expanded = df.loc[df.index.repeat(n_rep)].copy()

    # Assign correct years
    expanded["year"] = np.concatenate([
        np.arange(s, e + 1) for s, e in zip(start_year, end_year)
    ])

    expanded = expanded.reset_index(drop=True)

    # If the source data ends before the end of the vegetation timeseries, extendyears through 2024 to match vegetation timeseries
    if last_year < cn.years_annual[-1]:
        rows_2022 = expanded[expanded["year"] == 2022].copy()

        rows_2023 = rows_2022.copy()
        rows_2023["year"] = 2023

        rows_2024 = rows_2022.copy()
        rows_2024["year"] = 2024

        expanded = pd.concat([expanded, rows_2023, rows_2024], ignore_index=True)
        expanded = expanded.sort_values(series_cols + ["year"]).reset_index(drop=True)

    return expanded

In [5]:
# Combines 10x10 deg tile veg parquets into global long and wide tables using specified contextual layers
def aggreg_veg_parquets(parquet_files, agg_layers, LULUCF_component):

    # Sums value and area_ha by contextual layer combinations for each tile
    print("Simplifying each tile.")
    tile_aggs = []
    pre_agg_rows = 0
    for f in tqdm(parquet_files):
        df_tile = pd.read_parquet(f, columns=agg_layers + ['value', 'area_ha'])
        agg = (
            df_tile
            .groupby(agg_layers, dropna=False)[['value', 'area_ha']]
            .sum()
            .reset_index()
        )
        tile_aggs.append(agg)
        pre_agg_rows = pre_agg_rows + len(agg)
    print(f"Concatenated tile-level table has {pre_agg_rows}")

    # Sums across tiles, consolidating across combinations of contextual layers that repeat across tiles 
    print("Summing and simplifying all tiles")
    sums = (
        pd.concat(tile_aggs, ignore_index=True)
        .pipe(lambda df: df[~df['analysis_layer'].isin([
            'AGC_emission_factor_CO2_only__fraction',
            'removal_factor__AGC__MgC',
            # 'carbon_density__non_soil__MgC_ha',
        ])])
        .groupby(agg_layers, dropna=False)[['value', 'area_ha']]
        .sum()
        .reset_index()
        .sort_values(agg_layers)
        .reset_index(drop=True)
    )
    sums.rename(columns={'value': 'zonal_stats_sum'}, inplace=True)
    sums['LULUCF_component'] = LULUCF_component
    print(f"Simplified global table has shape {sums.shape}")
    # display(sums)

    return sums

In [6]:
# Converts long-format table to wide-format table.
# chunk_col argument breaks the processing up by values in the specified column because otherwise these tables are too large to pivot all at once
def long_to_wide(sums, agg_layers, LULUCF_component, chunk_col='cont_eco'):

    contextual_cols = [c for c in agg_layers if c != 'analysis_layer']

    # Collapse any duplicate contextual combinations before unstacking
    # (can arise when missing columns are filled with 'unassigned')
    print("  Collapsing any duplicate contextual combinations...")
    sums = (
        sums
        .groupby(contextual_cols + ['analysis_layer'], dropna=False)[['zonal_stats_sum', 'area_ha']]
        .sum()
        .reset_index()
    )


    print(f"  Pivoting flux values in chunks by {chunk_col}...")
    flux_chunks, area_chunks = [], []
    for val in tqdm(sums[chunk_col].unique()):
        chunk = sums[sums[chunk_col] == val]
        flux_chunks.append(
            chunk.set_index(contextual_cols + ['analysis_layer'])['zonal_stats_sum']
            .unstack('analysis_layer')
            .rename_axis(None, axis='columns')
        )
        area_chunks.append(
            chunk.set_index(contextual_cols + ['analysis_layer'])['area_ha']
            .unstack('analysis_layer')
            .add_suffix('__area_ha')
            .rename_axis(None, axis='columns')
        )

    print("  Concatenating and merging fluxes...")
    flux_wide = pd.concat(flux_chunks).fillna(0).reset_index()
    print("  Concatenating and merging areas...")
    area_wide = pd.concat(area_chunks).fillna(0).reset_index()
    print("  Merging...")
    sums_wide = flux_wide.merge(area_wide, on=contextual_cols)
    sums_wide['LULUCF_component'] = LULUCF_component
    del flux_chunks, area_chunks, flux_wide, area_wide
    print("  Done converting to wide")

    return sums_wide

In [8]:
# Contextual columns for making a unified wide table format

agg_layers = ['analysis_layer', 'year', 
              'land_state_node', 'land_state_meaning', 'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type',
              'adm0', 'country_name', 'region_L1', 'region_L2_L3',
              'continent', 'ecozone', 'continent_ecozone', 'climate_domain', 'cont_eco',
              # 'Landmark',
              'starting_composite_primary_forest',
              # 'KBA',
              # 'watershed', 'watershed_name',
              'drivers_of_TCL_1_km', 'driver_1km_text',
              'forest_age_category_end_of_interval',
              # 'gas',
              # 'WDPA', 'WDPA_type', 'WDPA_high_protection'
             ]

### Input table paths (zonal stats, and chunk stats for comparison purposes)

In [71]:
# Vegetation zonal stats output folder (separate parquet for each tile-- too large to combine into a single long-format parquet)
veg_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/vegetation_v{cn.veg_model_version_underscore}_standard_global__full_run_with_Canada_rerun__20260603/'

In [10]:
veg_parquet_files = sorted(glob.glob(f'{veg_zonal_stats_folder}*.parquet'))
# veg_parquet_files_subset = veg_parquet_files[0:5]
print(f"Found {len(veg_parquet_files)} parquet files")

Found 358 parquet files


In [12]:
# SOC (including mineral soil) zonal stats output
SOC_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/SOC_v{cn.SOC_model_version_underscore}_standard_global/'
SOC_parquet_name = f'SOC_zonal_stats_v{cn.SOC_model_version_underscore}_20260531_21_15_30.parquet'

In [13]:
# Organic soil zonal stats output
org_soil_zonal_stats_folder = f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/organic_soil_v1_0_1_standard_global__20260604/'
org_soil_parquet_name = 'master_zonal_full_disaggregation.parquet'  

In [14]:
# LULUCF outputs
LULUCF_zonal_stats_folder = Path(f'/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v{cn.LULUCF_model_version_underscore}__veg_v{cn.veg_model_version_underscore}__minsoil_v{cn.SOC_model_version_underscore}__orgsoil_v{cn.organic_soil_model_version_underscore}/')
LULUCF_zonal_stats_folder.mkdir(parents=True, exist_ok=True)
LULUCF_zonal_stats_folder

PosixPath('/mnt/c/GIS/AFOLU_flux_model/LULUCF/zonal_statistics/LULUCF_v1_0_0__veg_v1_0_5__minsoil_v1_0_1__orgsoil_v1_0_1')

In [44]:
# Vegetation chunk stats output (for comparison with zonal stats)
veg_chunk_stats_folder = f'/mnt/c/GIS/git/AFOLU_GHG_flux_model/chunk_stats/parquet_20260131_10_37_46__KEEP/'
veg_gross_outputs_parquet = f'{veg_chunk_stats_folder}vegetation_fluxes_20260131_10_37_28__v1_0_5__gross_outputs_1x1.parquet'
veg_net_outputs_parquet = f'{veg_chunk_stats_folder}vegetation_fluxes_20260131_10_37_28__v1_0_5__net_outputs_1x1.parquet'
veg_chunk_stats_gross = pd.read_parquet(veg_gross_outputs_parquet)
veg_chunk_stats_net = pd.read_parquet(veg_net_outputs_parquet)
# Combine gross and net chunk stats, aggregate to 10x10 deg tile level
veg_chunk_stats_combined = pd.concat([veg_chunk_stats_gross, veg_chunk_stats_net], ignore_index=True)

In [47]:
# SOC chunk stats output (for comparison with zonal stats)
SOC_chunk_stats_file = f'/mnt/c/GIS/git/AFOLU_GHG_flux_model/chunk_stats/KEEP_definitive_runs/SOC_density/v1_0_1__2000_2022_revised_org_soil__20260526/soil_carbon_densities_and_changes_1x1_chunk_statistics_20260526_15_43_37__KEEP.xlsx'
SOC_chunk_stats_gross = pd.read_excel(SOC_chunk_stats_file, sheet_name='other_outputs_1x1')
SOC_chunk_stats_net = pd.read_excel(SOC_chunk_stats_file, sheet_name='net_outputs_1x1')
# Combine gross and net chunk stats, aggregate to 10x10 deg tile level
SOC_chunk_stats_combined = pd.concat([SOC_chunk_stats_gross, SOC_chunk_stats_net], ignore_index=True)

### Global vegetation table with lots of contextual layers

In [ ]:
%%time

# Aggregation of tile-level zonal stats tables by most contextual layers 

veg_sum_all = aggreg_veg_parquets(veg_parquet_files, agg_layers, 'veg')
veg_sum_all.to_parquet(f'{veg_zonal_stats_folder}veg_v{cn.veg_model_version_underscore}__state_adm_conteco_prim_driver_age__long__{today}.parquet', index=False)
display(veg_sum_all)

In [ ]:
%%time

# Conversion of global vegetation table to wide foramt

# veg_sum_all = pd.read_parquet(f'{veg_zonal_stats_folder}veg_v{cn.veg_model_version_underscore}__state_adm_conteco_prim_driver_age__long__20260604.parquet')
veg_sum_all_wide = long_to_wide(veg_sum_all, agg_layers, 'veg', chunk_col='cont_eco')
# veg_sum_all_wide.to_parquet(f'{veg_zonal_stats_folder}veg_v{cn.veg_model_version_underscore}__state_adm_conteco_prim_driver_age__wide__{today}.parquet', index=False)
display(veg_sum_all_wide)

### Standardizing soil tables

In [ ]:
# %%time
# ### Standardizes SOC table in long format

# # Reads SOC zonal stats parquet table
# SOC_df_raw = pd.read_parquet(f'{SOC_zonal_stats_folder}{SOC_parquet_name}')

# # Renames value column with units
# SOC_df_raw.rename(columns={'value': 'zonal_stats_sum'}, inplace=True)

# # New column to specify this is mineral soil, as opposed to vegetation or organic soil (for combined LULUCF table).
# SOC_df_raw["LULUCF_component"] = "mineral_soil"

# print(f"Rows in SOC_df_raw: {len(SOC_df_raw)}")
# # SOC_df_raw


# # Drops all years before vegetation data (before 2016) because we don't need those for LULUCF (i.e. keeps only the 2020 and 2022 stock and change data)
# SOC_df_post_2016 = SOC_df_raw[SOC_df_raw["year"] >= cn.interval_end_years_annual[0]].reset_index(drop=True)
# print(f"Rows in SOC_df_post_2016: {len(SOC_df_post_2016)}")
# SOC_df_post_2016


# ## Fills in the years for multi-year interval SOC data.
# # 2020 is copied to 2016-2020 and 2022 to 2021-2024 (to match end of vegetation timeseries). 
# # If a year doesn't have data for a given year and combination of contextual layers, there is nothing to expand and all years in that interval are empty.
# series_cols = SOC_df_post_2016.drop(columns=['zonal_stats_sum', 'density__Mg_ha', 'area_ha']).columns.to_list()
# print(series_cols)
# SOC_df_post_2016_years_filled_in = fill_in_soil_years(SOC_df_post_2016, series_cols, agg_first=True)
# print(f"Rows in SOC_df_post_2016_years_filled_in: {len(SOC_df_post_2016_years_filled_in)}")
# # display(SOC_df_post_2016_years_filled_in)

# # Ratio below Should be very close to 4.5. If every contextual combination has every year, full year expansion would be 4.5 (2020 interval expanded 5x years, 2022 expanded 4x years).
# # However, some contextual combinations don't have all years (usually because they're so rare and other years have just 1 pixel of that combination),
# # so that doesn't get expanded to other years. This results in deviation from the 4.25x expansion. 
# print(f"Ratio of rows in unexpanded to expanded tables: {len(SOC_df_post_2016_years_filled_in)/len(SOC_df_post_2016)}")

# # Information on the years included in the table
# SOC_year_count = len(SOC_df_post_2016_years_filled_in['year'].unique())
# print(f"Years in SOC data: {SOC_df_post_2016_years_filled_in['year'].unique()} is {SOC_year_count} years")


# # Adds contextual columns found in the vegetation table
# contextual_cols = [c for c in agg_layers if c != 'analysis_layer']

# for col in contextual_cols:
#     if col not in SOC_df_post_2016_years_filled_in.columns:
#         SOC_df_post_2016_years_filled_in[col] = 'Unassigned'


# # QC: Timeseries of SOC change (full and mineral extent) to compare against chunk stats (Mg CO2/yr)
# # Some chunk stat values to check against: Density mineral extent 2020 = 350361333214; Net mineral extent 2020=1264590854; gain mineral extent 2020= -5495786224; loss mineral extent 2020= 6760376953
# layers = [
#     "SOC_density__mineral_soil_extent__0-30cm_MgC_ha",
#     "SOC_gain__mineral_soil_extent__0-30cm_MgCO2",
#     "SOC_loss__mineral_soil_extent__0-30cm_MgCO2",
#     "SOC_net__mineral_soil_extent__0-30cm_MgCO2"
# ]
# SOC_QC_sum = (
#     SOC_df_post_2016_years_filled_in[SOC_df_post_2016_years_filled_in["analysis_layer"].isin(layers)]
#     .groupby(["analysis_layer", "year"], as_index=False)["zonal_stats_sum"]
#     .sum()
# )
# # SOC_QC_sum

# SOC_df_post_2016_years_filled_in.to_parquet(f'{SOC_zonal_stats_folder}SOC_v{cn.SOC_model_version_underscore}__all_vars__long__{today}.parquet', index=False)
SOC_df_post_2016_years_filled_in = pd.read_parquet(f'{SOC_zonal_stats_folder}SOC_v{cn.SOC_model_version_underscore}__all_vars__long__20260604.parquet')
SOC_df_post_2016_years_filled_in

In [ ]:
# Converts SOC table from long to wide format

SOC_df_post_2016_years_filled_in_wide = long_to_wide(SOC_df_post_2016_years_filled_in, agg_layers, 'SOC', chunk_col='cont_eco')
SOC_df_post_2016_years_filled_in_wide.to_parquet(f'{SOC_zonal_stats_folder}SOC_v{cn.SOC_model_version_underscore}__all_vars__wide__{today}.parquet', index=False)
display(SOC_df_post_2016_years_filled_in_wide)

In [66]:
%%time
### Standardizes organic soil table (already in wide format)

# Reads organic soil zonal stats parquet table
org_soil_raw_wide = pd.read_parquet(f'{org_soil_zonal_stats_folder}{org_soil_parquet_name}')
org_soil_raw_wide
print(f"Rows in org_soil_raw_wide: {len(org_soil_raw_wide)}")

# Makes table columns generally match vegetation and SOC
org_soil_adjusted_wide = org_soil_raw_wide.drop(columns=['gadm_adm0', 'country', 'inventory_period', 'interval_start', 'land_use', 'drainage_class', 
                                                         'drained_state_meaning', 'burned_state_meaning', 'combined_state_nodes',
                                                         'drained_state_nodes', 'burned_state_nodes'])
display(org_soil_adjusted_wide.columns)

# Renames columns to match vegetation names
org_soil_adjusted_wide.rename(columns={'iso3': 'adm0', 
                                'component': 'analysis_layer',
                                'interval_end': 'year',
                                'river_basins': 'watershed',
                                'wdpa': 'WDPA',
                                'kba': 'KBA',
                                'area__ha': 'organic_soil__area_ha'},
                       inplace=True)

# Fills NaN (empty) values with 0s
org_soil_adjusted_wide = org_soil_adjusted_wide.fillna(0)
# org_soil_adjusted_wide

org_soil_adjusted_wide.columns = org_soil_adjusted_wide.columns.str.replace('_Mg_CO2', '__MgCO2')

# Adds attribute columns that are derived from existing columns. This is done in create_df() in zonal_stats_utilities.py for the vegetation and SOC data. 
org_soil_adjusted_wide['country_name'] = org_soil_adjusted_wide[cn.adm0_pattern].map(cn.iso_to_country)
org_soil_adjusted_wide['region_L1'] = org_soil_adjusted_wide[cn.adm0_pattern].map(cn.iso_to_region_UN_geoscheme_L1)
org_soil_adjusted_wide['region_L2_L3'] = org_soil_adjusted_wide[cn.adm0_pattern].map(cn.iso_to_region_UN_geoscheme_L2_L3)

# Because some rows for contextual layers may be blank
org_soil_adjusted_wide[cn.adm0_pattern] = org_soil_adjusted_wide[cn.adm0_pattern].fillna("Unassigned")
org_soil_adjusted_wide['country_name'] = org_soil_adjusted_wide['country_name'].fillna("Unassigned")
org_soil_adjusted_wide['region_L1'] = org_soil_adjusted_wide['region_L1'].fillna("Unassigned")
org_soil_adjusted_wide['region_L2_L3'] = org_soil_adjusted_wide['region_L2_L3'].fillna("Unassigned")

org_soil_adjusted_wide["climate_domain"] = org_soil_adjusted_wide["climate_domain"].replace({         
    "tropical": "Subtropical/tropical",
    "temperate": "Temperate",
    "boreal": "Boreal",
    "Unspecified": "Unassigned",
    "other_domain": "Unassigned"
})

org_soil_adjusted_wide['watershed_name'] = org_soil_adjusted_wide[cn.watersheds_pattern].map(cn.watershed_to_text)
org_soil_adjusted_wide["watershed_name"] = org_soil_adjusted_wide["watershed_name"].fillna("Unassigned")

org_soil_adjusted_wide['WDPA_type'] = org_soil_adjusted_wide[cn.WDPA_pattern].map(cn.WDPA_to_text)

# Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69aee45e-ce6c-8325-b1d0-a6c6b0e7ae2e
org_soil_adjusted_wide["WDPA_high_protection"] = "Other protection status"

org_soil_adjusted_wide.loc[org_soil_adjusted_wide["WDPA_type"] == "NA", "WDPA_high_protection"] = "Not protected"
org_soil_adjusted_wide.loc[org_soil_adjusted_wide["WDPA_type"].isin(["Category Ia", "Category Ib", "Category II", "Category III"]), "WDPA_high_protection"] = "High protection"

org_soil_adjusted_wide['driver_1km_text'] = org_soil_adjusted_wide[cn.drivers_of_loss_pattern].map(cn.drivers_to_text)
org_soil_adjusted_wide['driver_1km_text'] = org_soil_adjusted_wide['driver_1km_text'].fillna("Unassigned")
# org_soil_adjusted_wide

# Reaggregates by remaining contextual columns after dropped columns are removed
analysis_cols = [col for col in org_soil_adjusted_wide.columns if 'MgCO2' in col] + ['organic_soil__area_ha']
contextual_cols_adj = [col for col in org_soil_adjusted_wide.columns if col not in analysis_cols]
org_soil_adjusted_wide = (
    org_soil_adjusted_wide
    .groupby(contextual_cols_adj, dropna=False)[analysis_cols]
    .sum()
    .reset_index()
)
print(f"Rows in org_soil_adjusted_wide after reaggregation: {len(org_soil_adjusted_wide)}")

# Drops all years before vegetation data (before 2016) because we don't need those for LULUCF (i.e. keeps only the 2020 and 2022 stock and change data)
org_soil_post_2016_wide = org_soil_adjusted_wide[org_soil_adjusted_wide["year"] >= cn.interval_end_years_annual[0]].reset_index(drop=True)
print(f"Rows in org_soil_post_2016_wide: {len(org_soil_post_2016_wide)}")
org_soil_post_2016_wide


## Fills in the years for multi-year interval organic soil data.
# 2020 is copied to 2016-2020 and 2022 to 2021-2024 (to match end of vegetation timeseries). 
# If a year doesn't have data for a given year and combination of contextual layers, there is nothing to expand and all years in that interval are empty.
analysis_cols = [col for col in org_soil_post_2016_wide.columns if 'Mg_CO2' in col] + ['organic_soil__area_ha'] # Identifies non-contextual columns
contextual_cols = org_soil_post_2016_wide.drop(columns=analysis_cols).columns.to_list()
# display(series_cols)
org_soil_post_2016_filled_in_wide = fill_in_soil_years(org_soil_post_2016_wide, contextual_cols, agg_first=False)
print(f"Rows in org_soil_post_2016_filled_in_wide: {len(org_soil_post_2016_filled_in_wide)}")

# Ratio below Should be very close to 4.5. If every contextual combination has every year, full year expansion would be 4.5 (2020 interval expanded 5x years, 2022 expanded 4x years).
# However, some contextual combinations don't have all years (usually because they're so rare and other years have just 1 pixel of that combination),
# so that doesn't get expanded to other years. This results in deviation from the 4.25x expansion. 
print(f"Ratio of rows in unexpanded to expanded tables: {len(org_soil_post_2016_filled_in_wide)/len(org_soil_post_2016_wide)}")

# Information on the years included in the table
org_soil_year_count = len(org_soil_post_2016_filled_in_wide['year'].unique())
print(f"Years in SOC data: {org_soil_post_2016_filled_in_wide['year'].unique()} is {org_soil_year_count} years")

org_soil_post_2016_filled_in_wide.to_csv(f"{org_soil_zonal_stats_folder}org_soil__all_vars__wide__{today}.csv", index=False)
org_soil_post_2016_filled_in_wide.to_parquet(f"{org_soil_zonal_stats_folder}org_soil__all_vars__wide__{today}.parquet", index=False)
org_soil_post_2016_filled_in_wide

Rows in org_soil_raw_wide: 690580


Index(['interval_end', 'iso3', 'climate_domain', 'wdpa', 'landmark',
       'primary_forest_2001', 'kba', 'river_basins', 'drivers_of_TCL_1_km',
       'area__ha', 'drained_total_Mg_CO2e', 'drained_co2_onsite_Mg_CO2',
       'drained_co2_offsite_Mg_CO2', 'drained_total_co2_Mg_CO2',
       'drained_total_ch4_Mg_CO2e', 'drained_n2o_Mg_CO2e',
       'burned_total_Mg_CO2e', 'burned_total_co2_Mg_CO2',
       'burned_total_ch4_Mg_CO2e'],
      dtype='object')

Rows in org_soil_adjusted_wide after reaggregation: 391055
Rows in org_soil_post_2016_wide: 156422
CPU times: user 2.15 s, sys: 204 ms, total: 2.35 s
Wall time: 1.5 s


,year,adm0,climate_domain,WDPA,landmark,primary_forest_2001,KBA,watershed,drivers_of_TCL_1_km,country_name,region_L1,region_L2_L3,watershed_name,WDPA_type,WDPA_high_protection,driver_1km_text,drained_total__MgCO2e,drained_co2_onsite__MgCO2,drained_co2_offsite__MgCO2,drained_total_co2__MgCO2,drained_total_ch4__MgCO2e,drained_n2o__MgCO2e,burned_total__MgCO2e,burned_total_co2__MgCO2,burned_total_ch4__MgCO2e,organic_soil__area_ha
0,2020,ABW,Unassigned,0,0,0,0,0,0,Aruba,North America,Caribbean,Unassigned,NA,Not protected,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,161.661041
1,2020,ABW,Unassigned,0,0,0,0,0,6,Aruba,North America,Caribbean,Unassigned,NA,Not protected,Settlements and infrastructure,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,33.445736
2,2020,ABW,Unassigned,0,0,0,0,3001,0,Aruba,North America,Caribbean,Caribbean Coast,NA,Not protected,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12723.619141
3,2020,ABW,Unassigned,0,0,0,0,3001,6,Aruba,North America,Caribbean,Caribbean Coast,NA,Not protected,Settlements and infrastructure,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1327.724487
4,2020,ABW,Unassigned,0,0,0,1,0,0,Aruba,North America,Caribbean,Unassigned,NA,Not protected,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.555021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156417,2024,ZMB,Unassigned,11,1,0,1,7006,1,Zambia,Africa,Eastern Africa,Zambezi,Not Reported,Other protection status,Permanent agriculture,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,437.753296
156418,2024,ZMB,Unassigned,11,1,0,1,7006,5,Zambia,Africa,Eastern Africa,Zambezi,Not Reported,Other protection status,Wildfire,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.758027
156419,2024,ZMB,Unassigned,11,1,1,0,7005,0,Zambia,Africa,Eastern Africa,Congo,Not Reported,Other protection status,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,76.147049
156420,2024,ZMB,Unassigned,11,1,1,0,7005,1,Zambia,Africa,Eastern Africa,Congo,Not Reported,Other protection status,Permanent agriculture,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,337.026642


### Create combined LULUCF table (wide format)

In [15]:
# Reads wide component tables 
veg_sum_all_wide = pd.read_parquet(f'{veg_zonal_stats_folder}veg_v{cn.veg_model_version_underscore}__state_adm_conteco_prim_driver_age__wide__20260604.parquet')
SOC_df_post_2016_years_filled_in_wide = pd.read_parquet(f'{SOC_zonal_stats_folder}SOC_v{cn.SOC_model_version_underscore}__all_vars__wide__20260604.parquet')
org_soil_post_2016_filled_in_wide = pd.read_parquet(f"{org_soil_zonal_stats_folder}org_soil__all_vars__wide__20260604.parquet")

In [16]:
# Confirms that vegetation, SOC and organic soil tables have the right columns and the right options in each column
# Also, establishes contextual columns for each table

print(f"Columns in vegetation table are: {veg_sum_all_wide.columns}")
veg_analysis_cols = [col for col in veg_sum_all_wide.columns if 'MgC' in col] # Identifies non-contextual columns
veg_contextual_cols = veg_sum_all_wide.drop(columns=veg_analysis_cols).columns.to_list()  # Identifies contextual columns
print(f"Contextual columns in vegetation table are: {veg_contextual_cols}")
# for column in veg_contextual_cols:
#     print(f"{column}: {np.sort(veg_sum_all_wide[column].unique())}")

print("\n")
print(f"Columns in mineral soil table are: {SOC_df_post_2016_years_filled_in_wide.columns}")
SOC_analysis_cols = [col for col in SOC_df_post_2016_years_filled_in_wide.columns if 'MgC' in col] # Identifies non-contextual columns
SOC_contextual_cols = SOC_df_post_2016_years_filled_in_wide.drop(columns=SOC_analysis_cols).columns.to_list()  # Identifies contextual col
print(f"Contextual columns in SOC table are: {SOC_contextual_cols}")
# for column in SOC_contextual_cols:
#     print(f"{column}: {np.sort(SOC_df_post_2016_years_filled_in_wide[column].unique())}")

print("\n")
print(f"Columns in organic soil table are: {org_soil_post_2016_filled_in_wide.columns}")
org_soil_analysis_cols = [col for col in org_soil_post_2016_filled_in_wide.columns if 'MgC' in col] + ["organic_soil__area_ha"]  # Identifies non-contextual columns
org_soil_contextual_cols = org_soil_post_2016_filled_in_wide.drop(columns=org_soil_analysis_cols).columns.to_list()  # Identifies contextual col
print(f"Contextual columns in organic soil table are: {org_soil_contextual_cols}")
# for column in org_soil_contextual_cols:
#     print(f"{column}: {np.sort(org_soil_post_2016_filled_in_wide[column].unique())}")

Columns in vegetation table are: Index(['year', 'land_state_node', 'land_state_meaning',
       'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type',
       'adm0', 'country_name', 'region_L1', 'region_L2_L3', 'continent',
       'ecozone', 'continent_ecozone', 'climate_domain', 'cont_eco',
       'starting_composite_primary_forest', 'drivers_of_TCL_1_km',
       'driver_1km_text', 'forest_age_category_end_of_interval',
       'gross_emissions__AGC__MgCO2', 'gross_emissions__BGC__MgCO2',
       'gross_emissions__CH4__MgCO2e', 'gross_emissions__N2O__MgCO2e',
       'gross_emissions__all_C_pools__CO2_only__MgCO2',
       'gross_emissions__all_C_pools__all_gases__MgCO2e',
       'gross_emissions__all_C_pools__non_CO2_only__MgCO2e',
       'gross_emissions__deadwood_C__MgCO2',
       'gross_emissions__litter_C__MgCO2', 'gross_removals__AGC__MgCO2',
       'gross_removals__BGC__MgCO2', 'gross_removals__all_C_pools__MgCO2',
       'gross_removals__deadwood_C__MgCO2', 'gross

In [17]:
# Adds contextual columns from vegetation and mineral soil to organic soil table. All three tables should have the same contextual columns now.

org_soil_post_2016_standardized_wide = org_soil_post_2016_filled_in_wide.copy()

for col in veg_contextual_cols:
    if col not in org_soil_post_2016_standardized_wide.columns:
        org_soil_post_2016_standardized_wide[col] = 'Unassigned'
org_soil_post_2016_standardized_wide

,year,adm0,climate_domain,WDPA,landmark,primary_forest_2001,KBA,watershed,drivers_of_TCL_1_km,country_name,region_L1,region_L2_L3,watershed_name,WDPA_type,WDPA_high_protection,driver_1km_text,drained_total__MgCO2e,drained_co2_onsite__MgCO2,drained_co2_offsite__MgCO2,drained_total_co2__MgCO2,drained_total_ch4__MgCO2e,drained_n2o__MgCO2e,burned_total__MgCO2e,burned_total_co2__MgCO2,burned_total_ch4__MgCO2e,organic_soil__area_ha,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,tall_veg_type,continent,ecozone,continent_ecozone,cont_eco,starting_composite_primary_forest,forest_age_category_end_of_interval,LULUCF_component
0,2016,ABW,Unassigned,0,0,0,0,0,0,Aruba,North America,Caribbean,Unassigned,NA,Not protected,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,161.661041,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned
1,2017,ABW,Unassigned,0,0,0,0,0,0,Aruba,North America,Caribbean,Unassigned,NA,Not protected,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,161.661041,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned
2,2018,ABW,Unassigned,0,0,0,0,0,0,Aruba,North America,Caribbean,Unassigned,NA,Not protected,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,161.661041,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned
3,2019,ABW,Unassigned,0,0,0,0,0,0,Aruba,North America,Caribbean,Unassigned,NA,Not protected,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,161.661041,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned
4,2020,ABW,Unassigned,0,0,0,0,0,0,Aruba,North America,Caribbean,Unassigned,NA,Not protected,Unassigned,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,161.661041,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
703894,2024,ZMB,Unassigned,11,1,1,0,7005,1,Zambia,Africa,Eastern Africa,Congo,Not Reported,Other protection status,Permanent agriculture,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,337.026642,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned
703895,2021,ZMB,Unassigned,11,1,1,0,7005,3,Zambia,Africa,Eastern Africa,Congo,Not Reported,Other protection status,Shifting cultivation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.232221,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned
703896,2022,ZMB,Unassigned,11,1,1,0,7005,3,Zambia,Africa,Eastern Africa,Congo,Not Reported,Other protection status,Shifting cultivation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.232221,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned
703897,2023,ZMB,Unassigned,11,1,1,0,7005,3,Zambia,Africa,Eastern Africa,Congo,Not Reported,Other protection status,Shifting cultivation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.232221,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned,Unassigned


In [18]:
# Outer merge on shared contextual columns so all rows from all three tables are preserved
shared_contextual_cols = veg_contextual_cols
for col in shared_contextual_cols:
    veg_sum_all_wide[col]                           = veg_sum_all_wide[col].astype(str)
    SOC_df_post_2016_years_filled_in_wide[col]      = SOC_df_post_2016_years_filled_in_wide[col].astype(str)
    org_soil_post_2016_standardized_wide[col]       = org_soil_post_2016_standardized_wide[col].astype(str)

LULUCF_wide = (
    veg_sum_all_wide
    .merge(SOC_df_post_2016_years_filled_in_wide, on=shared_contextual_cols, how='outer')
    .merge(org_soil_post_2016_standardized_wide,      on=shared_contextual_cols, how='outer')
)

all_analysis_cols = veg_analysis_cols + SOC_analysis_cols + org_soil_analysis_cols
LULUCF_wide[all_analysis_cols] = LULUCF_wide[all_analysis_cols].fillna(0)

print(f"Rows in LULUCF_wide: {len(LULUCF_wide):,}")
display(LULUCF_wide)

Rows in LULUCF_wide: 3,552,496


,year,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,tall_veg_type,adm0,country_name,region_L1,region_L2_L3,continent,ecozone,continent_ecozone,climate_domain,cont_eco,starting_composite_primary_forest,drivers_of_TCL_1_km,driver_1km_text,forest_age_category_end_of_interval,gross_emissions__AGC__MgCO2,gross_emissions__BGC__MgCO2,gross_emissions__CH4__MgCO2e,gross_emissions__N2O__MgCO2e,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_emissions__deadwood_C__MgCO2,gross_emissions__litter_C__MgCO2,gross_removals__AGC__MgCO2,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2,gross_emissions__AGC__MgCO2__area_ha,gross_emissions__BGC__MgCO2__area_ha,gross_emissions__CH4__MgCO2e__area_ha,gross_emissions__N2O__MgCO2e__area_ha,gross_emissions__all_C_pools__CO2_only__MgCO2__area_ha,gross_emissions__all_C_pools__all_gases__MgCO2e__area_ha,gross_emissions__all_C_pools__non_CO2_only__MgCO2e__area_ha,gross_emissions__deadwood_C__MgCO2__area_ha,gross_emissions__litter_C__MgCO2__area_ha,gross_removals__AGC__MgCO2__area_ha,gross_removals__BGC__MgCO2__area_ha,gross_removals__all_C_pools__MgCO2__area_ha,gross_removals__deadwood_C__MgCO2__area_ha,gross_removals__litter_C__MgCO2__area_ha,net_flux__AGC__MgCO2__area_ha,net_flux__BGC__MgCO2__area_ha,net_flux__all_C_pools__CO2_only__MgCO2__area_ha,net_flux__all_C_pools__all_gases__MgCO2e__area_ha,net_flux__deadwood_C__MgCO2__area_ha,net_flux__litter_C__MgCO2__area_ha,LULUCF_component,SOC_density__full_extent__0-30cm_MgC_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha,SOC_gain__full_extent__0-30cm_MgCO2,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,SOC_loss__full_extent__0-30cm_MgCO2,SOC_loss__mineral_soil_extent__0-30cm_MgCO2,SOC_net__full_extent__0-30cm_MgCO2,SOC_net__mineral_soil_extent__0-30cm_MgCO2,SOC_density__full_extent__0-30cm_MgC_ha__area_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha__area_ha,SOC_gain__full_extent__0-30cm_MgCO2__area_ha,SOC_gain__mineral_soil_extent__0-30cm_MgCO2__area_ha,SOC_loss__full_extent__0-30cm_MgCO2__area_ha,SOC_loss__mineral_soil_extent__0-30cm_MgCO2__area_ha,SOC_net__full_extent__0-30cm_MgCO2__area_ha,SOC_net__mineral_soil_extent__0-30cm_MgCO2__area_ha,WDPA,landmark,primary_forest_2001,KBA,watershed,watershed_name,WDPA_type,WDPA_high_protection,drained_total__MgCO2e,drained_co2_onsite__MgCO2,drained_co2_offsite__MgCO2,drained_total_co2__MgCO2,drained_total_ch4__MgCO2e,drained_n2o__MgCO2e,burned_total__MgCO2e,burned_total_co2__MgCO2,burned_total_ch4__MgCO2e,organic_soil__area_ha
0,2016,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,mangrove,AGO,Angola,Africa,Middle Africa,Africa,Tropical dry forest,Africa-Tropical dry forest,Subtropical/tropical,1017,0,0,Unassigned,1_5yr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-167.707138,-48.635067,-262.444885,-43.268440,-2.834250,-167.707138,-48.635067,-262.444885,-262.444885,-43.268440,-2.834250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,veg,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2016,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,mangrove,AGO,Angola,Africa,Middle Africa,Africa,Tropical dry forest,Africa-Tropical dry forest,Subtropical/tropical,1017,0,1,Permanent agriculture,1_5yr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-118.736382,-34.433548,-185.810547,-30.633987,-2.006645,-118.736382,-34.433548,-185.810547,-185.810547,-30.633987,-2.00

In [19]:
# Counts the rows in the three wide component tables and the full LULUCF table. They should match since the three component tables share no contextual layer combinations

veg_table_shape = veg_sum_all_wide.shape
SOC_table_shape = SOC_df_post_2016_years_filled_in_wide.shape
org_soil_table_shape = org_soil_post_2016_standardized_wide.shape

indiv_table_row_sum = veg_table_shape[0] + SOC_table_shape[0] + org_soil_table_shape[0]
print(f"indiv_table_row_sum: {indiv_table_row_sum}")
print(f"LULUCF_wide.shape: {LULUCF_wide.shape}")

indiv_table_row_sum: 3552496
LULUCF_wide.shape: (3552496, 94)


In [20]:
# All columns in the LULUCF table
LULUCF_wide.columns

Index(['year', 'land_state_node', 'land_state_meaning',
       'land_state_broad_class', 'land_state_detailed_class', 'tall_veg_type',
       'adm0', 'country_name', 'region_L1', 'region_L2_L3', 'continent',
       'ecozone', 'continent_ecozone', 'climate_domain', 'cont_eco',
       'starting_composite_primary_forest', 'drivers_of_TCL_1_km',
       'driver_1km_text', 'forest_age_category_end_of_interval',
       'gross_emissions__AGC__MgCO2', 'gross_emissions__BGC__MgCO2',
       'gross_emissions__CH4__MgCO2e', 'gross_emissions__N2O__MgCO2e',
       'gross_emissions__all_C_pools__CO2_only__MgCO2',
       'gross_emissions__all_C_pools__all_gases__MgCO2e',
       'gross_emissions__all_C_pools__non_CO2_only__MgCO2e',
       'gross_emissions__deadwood_C__MgCO2',
       'gross_emissions__litter_C__MgCO2', 'gross_removals__AGC__MgCO2',
       'gross_removals__BGC__MgCO2', 'gross_removals__all_C_pools__MgCO2',
       'gross_removals__deadwood_C__MgCO2', 'gross_removals__litter_C__MgCO2',
    

In [22]:
# LULUCF-level gross and net sums

LULUCF_wide_with_LULUCF_cols = LULUCF_wide.copy()

LULUCF_wide_with_LULUCF_cols['organic_soil_emis_total__MgCO2e'] = LULUCF_wide_with_LULUCF_cols['drained_total__MgCO2e'] + LULUCF_wide_with_LULUCF_cols['burned_total__MgCO2e']
LULUCF_wide_with_LULUCF_cols['soil_emis_total__MgCO2e'] = LULUCF_wide_with_LULUCF_cols['organic_soil_emis_total__MgCO2e'] + LULUCF_wide_with_LULUCF_cols['SOC_loss__mineral_soil_extent__0-30cm_MgCO2']

LULUCF_wide_with_LULUCF_cols['LULUCF_gross_emissions__CO2__MgCO2'] = LULUCF_wide_with_LULUCF_cols['gross_emissions__all_C_pools__CO2_only__MgCO2'] + LULUCF_wide_with_LULUCF_cols['drained_total_co2__MgCO2'] + LULUCF_wide_with_LULUCF_cols['burned_total_co2__MgCO2']
LULUCF_wide_with_LULUCF_cols['LULUCF_gross_emissions__CH4__MgCO2e'] = LULUCF_wide_with_LULUCF_cols['gross_emissions__CH4__MgCO2e'] + LULUCF_wide_with_LULUCF_cols['drained_total_ch4__MgCO2e'] + LULUCF_wide_with_LULUCF_cols['burned_total_ch4__MgCO2e']
LULUCF_wide_with_LULUCF_cols['LULUCF_gross_emissions__N2O__MgCO2e'] = LULUCF_wide_with_LULUCF_cols['gross_emissions__N2O__MgCO2e'] + LULUCF_wide_with_LULUCF_cols['drained_n2o__MgCO2e']
LULUCF_wide_with_LULUCF_cols['LULUCF_gross_emissions__non_CO2__MgCO2e'] = LULUCF_wide_with_LULUCF_cols['LULUCF_gross_emissions__CH4__MgCO2e'] + LULUCF_wide_with_LULUCF_cols['LULUCF_gross_emissions__N2O__MgCO2e'] 
LULUCF_wide_with_LULUCF_cols['LULUCF_gross_emissions__all_gases__MgCO2e'] = LULUCF_wide_with_LULUCF_cols['gross_emissions__all_C_pools__all_gases__MgCO2e'] + LULUCF_wide_with_LULUCF_cols['soil_emis_total__MgCO2e']

LULUCF_wide_with_LULUCF_cols['LULUCF_gross_removals__MgCO2'] = LULUCF_wide_with_LULUCF_cols['gross_removals__all_C_pools__MgCO2'] + LULUCF_wide_with_LULUCF_cols['SOC_gain__mineral_soil_extent__0-30cm_MgCO2']
LULUCF_wide_with_LULUCF_cols['LULUCF_net_flux__MgCO2e'] = LULUCF_wide_with_LULUCF_cols['LULUCF_gross_emissions__all_gases__MgCO2e'] + LULUCF_wide_with_LULUCF_cols['LULUCF_gross_removals__MgCO2']
LULUCF_wide_with_LULUCF_cols.to_parquet(f'{LULUCF_zonal_stats_folder}/LULUCF_v{cn.LULUCF_model_version_underscore}__all_vars__wide__{today}.parquet', index=False)
LULUCF_wide_with_LULUCF_cols

,year,land_state_node,land_state_meaning,land_state_broad_class,land_state_detailed_class,tall_veg_type,adm0,country_name,region_L1,region_L2_L3,continent,ecozone,continent_ecozone,climate_domain,cont_eco,starting_composite_primary_forest,drivers_of_TCL_1_km,driver_1km_text,forest_age_category_end_of_interval,gross_emissions__AGC__MgCO2,gross_emissions__BGC__MgCO2,gross_emissions__CH4__MgCO2e,gross_emissions__N2O__MgCO2e,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_emissions__deadwood_C__MgCO2,gross_emissions__litter_C__MgCO2,gross_removals__AGC__MgCO2,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2,gross_emissions__AGC__MgCO2__area_ha,gross_emissions__BGC__MgCO2__area_ha,gross_emissions__CH4__MgCO2e__area_ha,gross_emissions__N2O__MgCO2e__area_ha,gross_emissions__all_C_pools__CO2_only__MgCO2__area_ha,gross_emissions__all_C_pools__all_gases__MgCO2e__area_ha,gross_emissions__all_C_pools__non_CO2_only__MgCO2e__area_ha,gross_emissions__deadwood_C__MgCO2__area_ha,gross_emissions__litter_C__MgCO2__area_ha,gross_removals__AGC__MgCO2__area_ha,gross_removals__BGC__MgCO2__area_ha,gross_removals__all_C_pools__MgCO2__area_ha,gross_removals__deadwood_C__MgCO2__area_ha,gross_removals__litter_C__MgCO2__area_ha,net_flux__AGC__MgCO2__area_ha,net_flux__BGC__MgCO2__area_ha,net_flux__all_C_pools__CO2_only__MgCO2__area_ha,net_flux__all_C_pools__all_gases__MgCO2e__area_ha,net_flux__deadwood_C__MgCO2__area_ha,net_flux__litter_C__MgCO2__area_ha,LULUCF_component,SOC_density__full_extent__0-30cm_MgC_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha,SOC_gain__full_extent__0-30cm_MgCO2,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,SOC_loss__full_extent__0-30cm_MgCO2,SOC_loss__mineral_soil_extent__0-30cm_MgCO2,SOC_net__full_extent__0-30cm_MgCO2,SOC_net__mineral_soil_extent__0-30cm_MgCO2,SOC_density__full_extent__0-30cm_MgC_ha__area_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha__area_ha,SOC_gain__full_extent__0-30cm_MgCO2__area_ha,SOC_gain__mineral_soil_extent__0-30cm_MgCO2__area_ha,SOC_loss__full_extent__0-30cm_MgCO2__area_ha,SOC_loss__mineral_soil_extent__0-30cm_MgCO2__area_ha,SOC_net__full_extent__0-30cm_MgCO2__area_ha,SOC_net__mineral_soil_extent__0-30cm_MgCO2__area_ha,WDPA,landmark,primary_forest_2001,KBA,watershed,watershed_name,WDPA_type,WDPA_high_protection,drained_total__MgCO2e,drained_co2_onsite__MgCO2,drained_co2_offsite__MgCO2,drained_total_co2__MgCO2,drained_total_ch4__MgCO2e,drained_n2o__MgCO2e,burned_total__MgCO2e,burned_total_co2__MgCO2,burned_total_ch4__MgCO2e,organic_soil__area_ha,organic_soil_emis_total__MgCO2e,soil_emis_total__MgCO2e,LULUCF_gross_emissions__CO2__MgCO2,LULUCF_gross_emissions__CH4__MgCO2e,LULUCF_gross_emissions__N2O__MgCO2e,LULUCF_gross_emissions__non_CO2__MgCO2e,LULUCF_gross_emissions__all_gases__MgCO2e,LULUCF_gross_removals__MgCO2,LULUCF_net_flux__MgCO2e
0,2016,11200000,"Gain of mangroves, no loss in interval",tree,tree_gain,mangrove,AGO,Angola,Africa,Middle Africa,Africa,Tropical dry forest,Africa-Tropical dry forest,Subtropical/tropical,1017,0,0,Unassigned,1_5yr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-167.707138,-48.635067,-262.444885,-43.268440,-2.834250,-167.707138,-48.635067,-262.444885,-262.444885,-43.268440,-2.834250,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,30.800205,veg,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,-262.444885,-262.444885
1,2016,11200000,"Gai

### QC: Checking if values in final LULUCF table match original tables for each component (chunk stats for veg and SOC, input wide table for organic soil)

In [49]:
### Compares global annual veg fluxes in final LULUCF table against original vegetation chunk stats table.
### I expect that chunk stats and zonal stats will differ somewhat for emissions. Emissions chunk stats should be lower because when I ran veg v1.0.5 and got chunk stats, 
### chunk stats summing still didn't sum in chunks with NaN pixels and maybe had other issues.
### So, chunk stats should be lower than zonal stats for emissions in general.
### Also, chunk stats uses the original global run of v1.0.5, which doesn't include the re-run tiles in Canada that included the missing emission factor for partial disturbance,
### so chunk stats also uses slightly older data that is actually missing emissions compared to zonal stats (that integrates the Canada re-run). 

# Chunk stats summed across all tiles
veg_chunk_tile_agg = (
    veg_chunk_stats_combined
    .groupby(['pattern', 'years'], dropna=False)['sum_value']
    .sum()
    .reset_index()
    .rename(columns={'years': 'year', 'pattern': 'variable', 'sum_value': 'tile_sum'})
)
veg_chunk_tile_agg['year'] = veg_chunk_tile_agg['year'].astype(int)
veg_chunk_tile_agg.rename(columns={'variable': 'analysis_layer', 'tile_sum': 'chunk_stats_sum'}, inplace=True)
veg_chunk_tile_agg

# Reference: veg chunk stats -- sum across all tiles by year and analysis_layer
chunk_veg_ref = (
    veg_chunk_tile_agg
    .groupby(['year', 'analysis_layer'])['chunk_stats_sum']
    .sum()
    .unstack('analysis_layer')
    .rename_axis(None, axis='columns')
    .rename(columns=lambda col: col.replace('_ha_yr', ''))
    .sort_index()
)
chunk_veg_ref.index = chunk_veg_ref.index.astype(int)
print("chunk stats")
display(chunk_veg_ref)

# From LULUCF_wide -- veg flux columns only, summed globally by year
flux_cols_veg = [col for col in veg_analysis_cols if '__area_ha' not in col]
LULUCF_veg_by_year = (
    LULUCF_wide
    .groupby('year')[flux_cols_veg]
    .sum()
    .sort_index()
)
LULUCF_veg_by_year.index = LULUCF_veg_by_year.index.astype(int)
print("final zonal stats")
display(LULUCF_veg_by_year)

# Only compare columns that exist in both (chunk stats may not have all layers)
shared_cols = [col for col in flux_cols_veg if col in chunk_veg_ref.columns]
print(f"Comparing {len(shared_cols)} shared analysis layers")
print(f"Layers in LULUCF_wide but not chunk stats: {set(flux_cols_veg) - set(chunk_veg_ref.columns)}")
print(f"Layers in chunk stats but not LULUCF_wide: {set(chunk_veg_ref.columns) - set(flux_cols_veg)}")

diff = LULUCF_veg_by_year[shared_cols] - chunk_veg_ref[shared_cols]
pct_diff = (diff / chunk_veg_ref[shared_cols].abs() * 100).round(4)

print("\n=== Absolute difference (zonal stats minus chunk stats) ===")
display(diff)
print("\n=== Percent difference (positive is zonal stats being > chunk stats) ===")
display(pct_diff)

chunk stats


,gross_emissions__AGC__MgCO2,gross_emissions__BGC__MgCO2,gross_emissions__CH4__MgCO2e,gross_emissions__N2O__MgCO2e,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_emissions__deadwood_C__MgCO2,gross_emissions__litter_C__MgCO2,gross_removals__AGC__MgCO2,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2
year,,,,,,,,,,,,,,,,,,,,
2016,8.535058e+09,1.741605e+09,2.775354e+08,1.045844e+08,1.041336e+10,1.079473e+10,3.821197e+08,1.088619e+08,2.794827e+07,-1.364858e+10,-4.353487e+09,-1.800743e+10,-4.403267e+06,-9.604854e+05,-4.994548e+09,-2.573350e+09,-7.436553e+09,-7.055180e+09,1.044586e+08,2.698779e+07
2017,7.968987e+09,1.725076e+09,2.604149e+08,9.995430e+07,9.820066e+09,1.017925e+10,3.603692e+08,9.970802e+07,2.636311e+07,-1.388448e+10,-4.349944e+09,-1.824217e+10,-6.176434e+06,-1.569487e+06,-5.804891e+09,-2.587675e+09,-8.274251e+09,-7.915072e+09,9.353158e+07,2.479362e+07
2018,8.017658e+09,1.815720e+09,2.203331e+08,8.761118e+07,9.962503e+09,1.026996e+10,3.079443e+08,1.019373e+08,2.728656e+07,-1.395070e+10,-4.377353e+09,-1.834200e+10,-1.084028e+07,-3.111306e+06,-5.823233e+09,-2.526370e+09,-8.234318e+09,-7.926866e+09,9.109702e+07,2.417525e+07
2019,8.496905e+09,1.795676e+09,2.798599e+08,1.064372e+08,1.042994e+10,1.081482e+10,3.862971e+08,1.091343e+08,2.832874e+07,-1.393895e+10,-4.384939e+09,-1.834378e+10,-1.525940e+07,-4.631280e+06,-5.325289e+09,-2.548510e+09,-7.756142e+09,-7.371266e+09,9.387486e+07,2.369746e+07
2020,9.722788e+09,2.053303e+09,2.873274e+08,1.142103e+08,1.192565e+10,1.232710e+10,4.015376e+08,1.182651e+08,3.133567e+07,-1.399504e+10,-4.418457e+09,-1.843908e+10,-1.949703e+07,-6.082023e+06,-4.159619e+09,-2.330530e+09,-6.365935e+09,-5.964480e+09,9.876806e+07,2.525365e+07
2021,8.861793e+09,1.913056e+09,2.402465e+08,9.696015e+07,1.092205e+10,1.125891e+10,3.372066e+08,1.156300e+08,3.163378e+07,-1.407045e+10,-4.468118e+09,-1.857016e+10,-2.388733e+07,-7.706297e+06,-5.103570e+09,-2.520500e+09,-7.508162e+09,-7.171299e+09,9.174268e+07,2.392749e+07
2022,8.165769e+09,1.899829e+09,2.085979e+08,8.142095e+07,1.021317e+10,1.050219e+10,2.900189e+08,1.169966e+08,3.063086e+07,-1.409446e+10,-4.448691e+09,-1.858058e+10,-2.809574e+07,-9.332863e+06,-5.823341e+09,-2.510818e+09,-8.223661e+09,-7.934641e+09,8.890082e+07,2.129800e+07
2023,8.215347e+09,1.773879e+09,2.181223e+08,8.968381e+07,1.012555e+10,1.043255e+10,3.078061e+08,1.078686e+08,2.848139e+07,-1.399200e+10,-4.450622e+09,-1.848580e+10,-3.220468e+07,-1.097158e+07,-5.663073e+09,-2.615745e+09,-8.185195e+09,-7.878187e+09,7.566393e+07,1.750981e+07
2024,8.991911e+09,1.891452e+09,2.683090e+08,1.034220e+08,1.102083e+10,1.139186e+10,3.717310e+08,1.084914e+08,2.902831e+07,-1.441729e+10,-4.664952e+09,-1.913318e+10,-3.771986e+07,-1.321706e+07,-5.324484e+09,-2.721164e+09,-7.958425e+09,-7.587393e+09,7.077155e+07,1.581125e+07


final zonal stats


,gross_emissions__AGC__MgCO2,gross_emissions__BGC__MgCO2,gross_emissions__CH4__MgCO2e,gross_emissions__N2O__MgCO2e,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_emissions__deadwood_C__MgCO2,gross_emissions__litter_C__MgCO2,gross_removals__AGC__MgCO2,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2
year,,,,,,,,,,,,,,,,,,,,
2016,8.579622e+09,1.751007e+09,277535360.0,104584352.0,1.046744e+10,1.084956e+10,382119712.0,108861864.0,27948272.0,-1.364858e+10,-4.353487e+09,-1.800743e+10,-4403267.0,-9.604854e+05,-5.068954e+09,-2.602480e+09,-7.539987e+09,-7.157868e+09,104458600.0,26987786.0
2017,8.011849e+09,1.729690e+09,260927200.0,100244312.0,9.867610e+09,1.022878e+10,361171520.0,99708016.0,26363110.0,-1.388448e+10,-4.349944e+09,-1.824217e+10,-6176434.0,-1.569486e+06,-5.872635e+09,-2.620254e+09,-8.374564e+09,-8.013393e+09,93531584.0,24793624.0
2018,8.054772e+09,1.823392e+09,220547008.0,87731336.0,1.000739e+10,1.031567e+10,308278336.0,101937288.0,27286558.0,-1.395070e+10,-4.377353e+09,-1.834200e+10,-10840277.0,-3.111306e+06,-5.895928e+09,-2.553961e+09,-8.334617e+09,-8.026338e+09,91097016.0,24175252.0
2019,8.615472e+09,1.806362e+09,282615168.0,107982656.0,1.055930e+10,1.094989e+10,390597824.0,109134264.0,28328744.0,-1.393895e+10,-4.384940e+09,-1.834378e+10,-15259397.0,-4.631280e+06,-5.323476e+09,-2.578577e+09,-7.784481e+09,-7.393883e+09,93874872.0,23697464.0
2020,9.749906e+09,2.061720e+09,287361312.0,114229312.0,1.196123e+10,1.236282e+10,401590624.0,118265096.0,31335668.0,-1.399504e+10,-4.418457e+09,-1.843908e+10,-19497034.0,-6.082023e+06,-4.245134e+09,-2.356737e+09,-6.477849e+09,-6.076258e+09,98768064.0,25253646.0
2021,8.904365e+09,1.918495e+09,241109024.0,97445360.0,1.097012e+10,1.130868e+10,338554400.0,115630016.0,31633782.0,-1.407045e+10,-4.468119e+09,-1.857016e+10,-23887334.0,-7.706297e+06,-5.166082e+09,-2.549624e+09,-7.600035e+09,-7.261480e+09,91742680.0,23927484.0
2022,8.261025e+09,1.912891e+09,210173600.0,82306328.0,1.032154e+10,1.061402e+10,292479936.0,116996568.0,30630864.0,-1.409446e+10,-4.448691e+09,-1.858058e+10,-28095742.0,-9.332863e+06,-5.833435e+09,-2.535800e+09,-8.259036e+09,-7.966556e+09,88900824.0,21298002.0
2023,8.695273e+09,1.807633e+09,235021936.0,99210088.0,1.063926e+10,1.097349e+10,334232032.0,107868616.0,28481388.0,-1.399200e+10,-4.450622e+09,-1.848580e+10,-32204682.0,-1.097158e+07,-5.296731e+09,-2.642989e+09,-7.846546e+09,-7.512314e+09,75663928.0,17509806.0
2024,9.383472e+09,1.959626e+09,275464384.0,107443600.0,1.148062e+10,1.186353e+10,382908000.0,108491408.0,29028308.0,-1.441729e+10,-4.664954e+09,-1.913318e+10,-37719856.0,-1.321706e+07,-5.033815e+09,-2.705328e+09,-7.652560e+09,-7.269652e+09,70771552.0,15811253.0


Comparing 20 shared analysis layers
Layers in LULUCF_wide but not chunk stats: set()
Layers in chunk stats but not LULUCF_wide: set()

=== Absolute difference (zonal stats minus chunk stats) ===


,gross_emissions__AGC__MgCO2,gross_emissions__BGC__MgCO2,gross_emissions__CH4__MgCO2e,gross_emissions__N2O__MgCO2e,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_emissions__deadwood_C__MgCO2,gross_emissions__litter_C__MgCO2,gross_removals__AGC__MgCO2,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2
year,,,,,,,,,,,,,,,,,,,,
2016,4.456466e+07,9.401758e+06,-1.416174e+01,1.868576e+00,5.407961e+07,5.482603e+07,1.209413e+00,-3.936049,0.290834,-440.199387,-756.349037,-594.562210,0.034605,-0.012276,-7.440602e+07,-2.912983e+07,-1.034340e+08,-1.026875e+08,1.416415,-0.438643
2017,4.286203e+07,4.614646e+06,5.123292e+05,2.900105e+05,4.754423e+07,4.953668e+07,8.023622e+05,-1.787626,-0.179188,568.863472,-223.213688,-1401.671268,-0.219645,0.027064,-6.774436e+07,-3.257939e+07,-1.003133e+08,-9.832119e+07,3.311615,0.806339
2018,3.711405e+07,7.671928e+06,2.139295e+05,1.201574e+05,4.488554e+07,4.571138e+07,3.340791e+05,-3.098585,1.442337,476.958036,84.626007,-640.922913,-0.281041,-0.173661,-7.269495e+07,-2.759134e+07,-1.002982e+08,-9.947191e+07,0.032215,-0.089758
2019,1.185674e+08,1.068557e+07,2.755251e+06,1.545502e+06,1.293545e+08,1.350756e+08,4.300747e+06,-5.893616,-0.836670,468.055357,-702.275763,-1628.096748,-0.464604,0.024127,1.812915e+06,-3.006769e+07,-2.833833e+07,-2.261751e+07,10.957322,0.191419
2020,2.711893e+07,8.416957e+06,3.396167e+04,1.902289e+04,3.558023e+07,3.571606e+07,5.300319e+04,-10.283278,-0.229969,514.770365,257.442824,549.732521,-0.845283,-0.066000,-8.551470e+07,-2.620685e+07,-1.119145e+08,-1.117787e+08,1.795521,-0.302151
2021,4.257227e+07,5.439596e+06,8.625641e+05,4.852111e+05,4.807817e+07,4.977037e+07,1.347772e+06,0.598167,0.474239,-182.757067,-786.848658,-1248.633049,-0.475751,0.292904,-6.251194e+07,-2.912381e+07,-9.187293e+07,-9.018131e+07,1.150030,-1.232373
2022,9.525604e+07,1.306238e+07,1.575664e+06,8.853781e+05,1.083682e+08,1.118287e+08,2.461052e+06,-0.328157,-0.040244,47.409245,-693.691424,-1781.394951,-0.448942,-0.355436,-1.009394e+07,-2.498243e+07,-3.537532e+07,-3.191507e+07,7.044931,1.412588
2023,4.799265e+08,3.375464e+07,1.689968e+07,9.526274e+06,5.137113e+08,5.409363e+08,2.642597e+07,6.028562,-0.180576,-44.230682,83.933264,-172.133297,-0.326249,-0.248490,3.663427e+08,-2.724419e+07,3.386490e+08,3.658736e+08,0.792044,-0.493061
2024,3.915611e+08,6.817393e+07,7.155382e+06,4.021605e+06,4.597879e+08,4.716636e+08,1.117699e+07,0.933281,-0.595469,-421.655617,-1605.693224,-2819.204136,-0.589298,0.589218,2.906699e+08,1.583541e+07,3.058653e+08,3.177409e+08,-1.688184,0.233926



=== Percent difference (positive is zonal stats being > chunk stats) ===


,gross_emissions__AGC__MgCO2,gross_emissions__BGC__MgCO2,gross_emissions__CH4__MgCO2e,gross_emissions__N2O__MgCO2e,gross_emissions__all_C_pools__CO2_only__MgCO2,gross_emissions__all_C_pools__all_gases__MgCO2e,gross_emissions__all_C_pools__non_CO2_only__MgCO2e,gross_emissions__deadwood_C__MgCO2,gross_emissions__litter_C__MgCO2,gross_removals__AGC__MgCO2,gross_removals__BGC__MgCO2,gross_removals__all_C_pools__MgCO2,gross_removals__deadwood_C__MgCO2,gross_removals__litter_C__MgCO2,net_flux__AGC__MgCO2,net_flux__BGC__MgCO2,net_flux__all_C_pools__CO2_only__MgCO2,net_flux__all_C_pools__all_gases__MgCO2e,net_flux__deadwood_C__MgCO2,net_flux__litter_C__MgCO2
year,,,,,,,,,,,,,,,,,,,,
2016,0.5221,0.5398,-0.0000,0.0000,0.5193,0.5079,0.0000,-0.0,0.0,-0.0,-0.0,-0.0,0.0,-0.0,-1.4897,-1.1320,-1.3909,-1.4555,0.0,-0.0
2017,0.5379,0.2675,0.1967,0.2901,0.4842,0.4866,0.2227,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,-1.1670,-1.2590,-1.2124,-1.2422,0.0,0.0
2018,0.4629,0.4225,0.0971,0.1371,0.4505,0.4451,0.1085,-0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,-1.2484,-1.0921,-1.2181,-1.2549,0.0,-0.0
2019,1.3954,0.5951,0.9845,1.4520,1.2402,1.2490,1.1133,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,0.0,0.0340,-1.1798,-0.3654,-0.3068,0.0,0.0
2020,0.2789,0.4099,0.0118,0.0167,0.2984,0.2897,0.0132,-0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-2.0558,-1.1245,-1.7580,-1.8741,0.0,-0.0
2021,0.4804,0.2843,0.3590,0.5004,0.4402,0.4421,0.3997,0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.0,-1.2249,-1.1555,-1.2236,-1.2575,0.0,-0.0
2022,1.1665,0.6876,0.7554,1.0874,1.0611,1.0648,0.8486,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,-0.0,-0.1733,-0.9950,-0.4302,-0.4022,0.0,0.0
2023,5.8418,1.9029,7.7478,10.6221,5.0734,5.1851,8.5853,0.0,-0.0,-0.0,0.0,-0.0,-0.0,-0.0,6.4690,-1.0415,4.1373,4.6441,0.0,-0.0
2024,4.3546,3.6043,2.6668,3.8885,4.1720,4.1404,3.0067,0.0,-0.0,-0.0,-0.0,-0.0,-0.0,0.0,5.4591,0.5819,3.8433,4.1877,-0.0,0.0


In [60]:
### Compares global annual SOC fluxes in final LULUCF table against original SOC chunk stats table
### I expect that SOC chunk stats and zonal stats should be pretty much identical. 

# Chunk stats summed across all tiles
SOC_chunk_tile_agg = (
    SOC_chunk_stats_combined
    .groupby(['pattern', 'years'], dropna=False)['sum_value']
    .sum()
    .reset_index()
    .rename(columns={'years': 'year', 'pattern': 'variable', 'sum_value': 'tile_sum'})
)
SOC_chunk_tile_agg['year'] = SOC_chunk_tile_agg['year'].astype(int)
SOC_chunk_tile_agg.rename(columns={'variable': 'analysis_layer', 'tile_sum': 'chunk_stats_sum'}, inplace=True)

# Final LULUCF table has only post-2016 soil data, so need to limit chunk stats for comparison
SOC_chunk_tile_agg_post_2016 = SOC_chunk_tile_agg[SOC_chunk_tile_agg["year"] >= cn.interval_end_years_annual[0]].reset_index(drop=True)
SOC_chunk_tile_agg_post_2016

# Reference: SOC chunk stats -- sum across all tiles by year and analysis_layer
chunk_SOC_ref = (
    SOC_chunk_tile_agg_post_2016
    .groupby(['year', 'analysis_layer'])['chunk_stats_sum']
    .sum()
    .unstack('analysis_layer')
    .rename_axis(None, axis='columns')
    .rename(columns=lambda col: col.replace('_ha_yr', ''))
    .sort_index()
)
chunk_SOC_ref.index = chunk_SOC_ref.index.astype(int)
print("chunk stats")
display(chunk_SOC_ref)

# From LULUCF_wide -- veg flux columns only, summed globally by year
flux_cols_SOC = [col for col in SOC_analysis_cols if '__area_ha' not in col]
LULUCF_SOC_by_year = (
    LULUCF_wide
    .groupby('year')[flux_cols_SOC]
    .sum()
    .sort_index()
)
LULUCF_SOC_by_year.index = LULUCF_SOC_by_year.index.astype(int)

# Limits SOC from LULUCF table to original years
LULUCF_SOC_by_year = LULUCF_SOC_by_year.loc[[2020, 2022]]

print("final zonal stats")
display(LULUCF_SOC_by_year)

# Only compare columns that exist in both (chunk stats may not have all layers)
shared_cols = [col for col in flux_cols_SOC if col in chunk_SOC_ref.columns]
print(f"Comparing {len(shared_cols)} shared analysis layers")
print(f"Layers in LULUCF_wide but not chunk stats: {set(flux_cols_SOC) - set(chunk_SOC_ref.columns)}")
print(f"Layers in chunk stats but not LULUCF_wide: {set(chunk_SOC_ref.columns) - set(flux_cols_SOC)}")

diff = LULUCF_SOC_by_year[shared_cols] - chunk_SOC_ref[shared_cols]
pct_diff = (diff / chunk_SOC_ref[shared_cols].abs() * 100).round(8)

print("\n=== Absolute difference (zonal stats minus chunk stats) ===")
display(diff)
print("\n=== Percent difference (positive is zonal stats being > chunk stats) ===")
display(pct_diff)

chunk stats


,SOC_density__full_extent__0-30cm_MgC_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha,SOC_gain__full_extent__0-30cm_MgCO2,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,SOC_loss__full_extent__0-30cm_MgCO2,SOC_loss__mineral_soil_extent__0-30cm_MgCO2,SOC_net__full_extent__0-30cm_MgCO2,SOC_net__mineral_soil_extent__0-30cm_MgCO2
year,,,,,,,,
2020,4.446701e+11,3.503613e+11,-7.063454e+09,-5.495786e+09,8.951121e+09,6.760377e+09,1.887667e+09,1.264591e+09
2022,4.436921e+11,3.491311e+11,-1.909820e+10,-1.368484e+10,2.088861e+10,1.593772e+10,1.790410e+09,2.252878e+09


final zonal stats


,SOC_density__full_extent__0-30cm_MgC_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha,SOC_gain__full_extent__0-30cm_MgCO2,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,SOC_loss__full_extent__0-30cm_MgCO2,SOC_loss__mineral_soil_extent__0-30cm_MgCO2,SOC_net__full_extent__0-30cm_MgCO2,SOC_net__mineral_soil_extent__0-30cm_MgCO2
year,,,,,,,,
2020,4.446701e+11,3.503614e+11,-7.063455e+09,-5.495786e+09,8.951121e+09,6.760377e+09,1.887667e+09,1.264591e+09
2022,4.436922e+11,3.491311e+11,-1.909820e+10,-1.368484e+10,2.088861e+10,1.593772e+10,1.790410e+09,2.252878e+09


Comparing 8 shared analysis layers
Layers in LULUCF_wide but not chunk stats: set()
Layers in chunk stats but not LULUCF_wide: set()

=== Absolute difference (zonal stats minus chunk stats) ===


,SOC_density__full_extent__0-30cm_MgC_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha,SOC_gain__full_extent__0-30cm_MgCO2,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,SOC_loss__full_extent__0-30cm_MgCO2,SOC_loss__mineral_soil_extent__0-30cm_MgCO2,SOC_net__full_extent__0-30cm_MgCO2,SOC_net__mineral_soil_extent__0-30cm_MgCO2
year,,,,,,,,
2020,14497.624390,21025.929932,-473.062742,-271.633113,-100.668669,391.475938,165.407407,-5.609629
2022,9551.541443,-4073.598816,-996.286205,-284.698271,-294.545761,262.528809,38.123938,4.638460



=== Percent difference (positive is zonal stats being > chunk stats) ===


,SOC_density__full_extent__0-30cm_MgC_ha,SOC_density__mineral_soil_extent__0-30cm_MgC_ha,SOC_gain__full_extent__0-30cm_MgCO2,SOC_gain__mineral_soil_extent__0-30cm_MgCO2,SOC_loss__full_extent__0-30cm_MgCO2,SOC_loss__mineral_soil_extent__0-30cm_MgCO2,SOC_net__full_extent__0-30cm_MgCO2,SOC_net__mineral_soil_extent__0-30cm_MgCO2
year,,,,,,,,
2020,0.000003,0.000006,-0.000007,-0.000005,-0.000001,0.000006,0.000009,-4.400000e-07
2022,0.000002,-0.000001,-0.000005,-0.000002,-0.000001,0.000002,0.000002,2.100000e-07


In [70]:
### Compares global annual organic soil fluxes in final LULUCF table against original original table
### I expect that the original and final tables should be identical. 

# Reference: original organic soil table summed by year
flux_cols_org_soil = [col for col in org_soil_analysis_cols 
                      if '__area_ha' not in col and col != 'organic_soil__area_ha']

org_soil_ref_by_year = (
    org_soil_post_2016_wide
    .groupby('year')[flux_cols_org_soil]
    .sum()
    .sort_index()
)
org_soil_ref_by_year.index = org_soil_ref_by_year.index.astype(int)
print("chunk stats")
display(org_soil_ref_by_year)

# From LULUCF_wide -- organic soil flux columns, summed globally by year
LULUCF_org_soil_by_year = (
    LULUCF_wide
    .groupby('year')[flux_cols_org_soil]
    .sum()
    .sort_index()
)
LULUCF_org_soil_by_year.index = LULUCF_org_soil_by_year.index.astype(int)

# Limits organic soil from LULUCF table to original years
LULUCF_org_soil_by_year = LULUCF_org_soil_by_year.loc[[2020, 2024]]

print("final zonal stats")
display(LULUCF_org_soil_by_year)

shared_cols_org_soil = [col for col in flux_cols_org_soil if col in org_soil_ref_by_year.columns]
print(f"Comparing {len(shared_cols_org_soil)} shared organic soil columns")
print(f"Columns in LULUCF_wide but not org soil ref: {set(flux_cols_org_soil) - set(org_soil_ref_by_year.columns)}")
print(f"Columns in org soil ref but not LULUCF_wide: {set(org_soil_ref_by_year.columns) - set(flux_cols_org_soil)}")

diff = LULUCF_org_soil_by_year[shared_cols_org_soil] - org_soil_ref_by_year[shared_cols_org_soil]
pct_diff = (diff / org_soil_ref_by_year[shared_cols_org_soil].abs() * 100).round(4)

print("\n=== Absolute difference (LULUCF_wide minus original org soil) ===")
display(diff)
print("\n=== Percent difference ===")
display(pct_diff)

chunk stats


,drained_total__MgCO2e,drained_co2_onsite__MgCO2,drained_co2_offsite__MgCO2,drained_total_co2__MgCO2,drained_total_ch4__MgCO2e,drained_n2o__MgCO2e,burned_total__MgCO2e,burned_total_co2__MgCO2,burned_total_ch4__MgCO2e
year,,,,,,,,,
2020,2.038003e+09,1.701882e+09,1.012966e+08,1.803179e+09,8.882245e+07,1.460017e+08,4.434621e+08,3.593733e+08,8.408877e+07
2024,2.128826e+09,1.764331e+09,1.205195e+08,1.884850e+09,8.688165e+07,1.570943e+08,5.407290e+08,4.457287e+08,9.500038e+07


final zonal stats


,drained_total__MgCO2e,drained_co2_onsite__MgCO2,drained_co2_offsite__MgCO2,drained_total_co2__MgCO2,drained_total_ch4__MgCO2e,drained_n2o__MgCO2e,burned_total__MgCO2e,burned_total_co2__MgCO2,burned_total_ch4__MgCO2e
year,,,,,,,,,
2020,2.038003e+09,1.701882e+09,1.012966e+08,1.803179e+09,8.882245e+07,1.460017e+08,4.434621e+08,3.593733e+08,8.408877e+07
2024,2.128826e+09,1.764331e+09,1.205195e+08,1.884850e+09,8.688165e+07,1.570943e+08,5.407290e+08,4.457287e+08,9.500038e+07


Comparing 9 shared organic soil columns
Columns in LULUCF_wide but not org soil ref: set()
Columns in org soil ref but not LULUCF_wide: set()

=== Absolute difference (LULUCF_wide minus original org soil) ===


,drained_total__MgCO2e,drained_co2_onsite__MgCO2,drained_co2_offsite__MgCO2,drained_total_co2__MgCO2,drained_total_ch4__MgCO2e,drained_n2o__MgCO2e,burned_total__MgCO2e,burned_total_co2__MgCO2,burned_total_ch4__MgCO2e
year,,,,,,,,,
2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



=== Percent difference ===


,drained_total__MgCO2e,drained_co2_onsite__MgCO2,drained_co2_offsite__MgCO2,drained_total_co2__MgCO2,drained_total_ch4__MgCO2e,drained_n2o__MgCO2e,burned_total__MgCO2e,burned_total_co2__MgCO2,burned_total_ch4__MgCO2e
year,,,,,,,,,
2020,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Figure creation

In [ ]:
# General code for figures

years = len(cn.interval_end_years_annual)

gross_emis_all_gases_legend = "Gross emissions (all gases)"
gross_emis_CO2_legend = "Gross emissions (CO₂ only)"
gross_emis_non_CO2_legend = "Gross emissions (non-CO₂)"
net_flux_all_gases_legend = "Net flux (all gases)"
gross_removals_legend = "Gross removals"

broad_class_dict = {
    "tree": "Tall vegetation",
    "short_veg": "Short vegetation",
    "crop": "Cropland"
}

detailed_class_dict = {
    "tree_tree_undisturbed": "Undisturbed tall veg",
    "tree_tree_disturbed": "Partially disturbed tall veg", 
    "tree_tree_disturbed_fire_ony": "Tall veg with fire but not height reduction", 
    "tree_loss": "Tall veg loss",
    "tree_gain": "Tall veg gain",
    "short_veg_short_veg_undisturbed": "Stable short veg",
    "short_veg_gain": "Short veg gain",
    "short_veg_loss": "Short veg loss", 
    "crop_crop_undisturbed": "Stable cropland",
    "crop_gain": "Cropland gain",
    "crop_loss": "Cropland loss"
}

tall_veg_type_dict = {
    "mangrove": "Mangrove",
    "oil_palm": "Oil palm",
    "non_oil_palm_planted_trees": "Non-oil palm planted trees",
    "natural_tree_cover": "Natural tree cover",
    "trees_in_other_land_covers": "Trees in other land covers",
    "non_tall_vegetation": "Non-tall vegetation"
}

primary_forest_dict = {
    0: "Not primary forest",
    1: "Primary forest"
}

tall_veg_change_type_classes = [
    "Tall veg loss",
    "Tall veg gain",
    "Partially disturbed tall veg",
    "Undisturbed (except for fires) tall veg"
]

# Aggregates by analysis layer
def aggregate_layer(df, pattern, context_to_analyze):
    d = df[df["analysis_layer"] == pattern]
    d = (
        d.groupby([context_to_analyze, "year"], as_index=False)
         .agg({"value": "sum"})
    )
    d["value_Gt"] = d["value"] / 1e9
    return d

sns.set_theme(style="white", context="talk")

In [ ]:
### Global LULUCF components with gross as bars and net veg and LULUCF as lines (single panel)
### Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69c6d457-96b4-8328-9fca-ff50f8273233
### Colors consistent with LULUCF maps from Claude (session 'LULUCF graph color consistency with maps')

df_fig = LULUCF_outputs_dropped.copy()

## Various lists and dictionaries needed to create the figure

# Drop unwanted vegetation-only emissions layers
layers_to_drop = [
    f"veg_{cn.gross_emis_all_C_pools_CO2_only_pattern}",
    f"veg_{cn.gross_emis_all_C_pools_non_CO2_only_pattern}",
]

# Net flux layers to draw as lines instead of bars
net_flux_layers = [
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
    "LULUCF_net_flux__MgCO2e",
]

# Organic soil layers to combine for graphing
organic_soil_layers = [
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]

organic_soil_combined_label = "organic_soil__all_gases__MgCO2e"

desired_bar_order = [
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2",
    "organic_soil__all_gases__MgCO2e",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}",
    f"veg_{cn.gross_removals_all_C_pools_pattern}",
]

desired_line_order = [
    "LULUCF_net_flux__MgCO2e",
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
]

color_map = {
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2": "#8c510a",        # emissions idx 8 (medium-dark brown)
    "organic_soil__all_gases__MgCO2e":               "#543005",         # emissions idx 9 (darkest brown)
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}": "#dfc27d",   # emissions idx 6 (medium tan)
    f"veg_{cn.gross_removals_all_C_pools_pattern}":  "#35978f",         # removals  idx 2 (medium teal)
    "LULUCF_net_flux__MgCO2e":                       "#000000",         # black line
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}": "#666633",  # green
}

label_map = {
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2":
        "Mineral soil carbon change",
    "organic_soil__all_gases__MgCO2e":
        "Organic soil emissions",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}":
        "Vegetation gross emissions",
    f"veg_{cn.gross_removals_all_C_pools_pattern}":
        "Vegetation gross removals",
    "LULUCF_net_flux__MgCO2e":
        "LULUCF net flux",
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}":
    #     "Vegetation net flux",
}


## Creates the figure

df_fig = df_fig[~df_fig["analysis_layer"].isin(layers_to_drop)].copy()

# Combine the three organic soil layers into one label
df_fig["analysis_layer"] = df_fig["analysis_layer"].replace(
    {layer: organic_soil_combined_label for layer in organic_soil_layers}
)

# Sum by year and analysis layer
df_ts = (
    df_fig
    .groupby(["year", "analysis_layer"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)

# Convert to Gt CO2e/yr
df_ts["flux_Gt_CO2e_yr"] = df_ts["flux_Mg_CO2e_yr"] / 1e9

# Pivot to wide form
df_wide = (
    df_ts
    .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
    .fillna(0)
)

# Apply ordering (keep only columns that exist)
bar_cols = [c for c in desired_bar_order if c in df_wide.columns]
line_cols = [c for c in desired_line_order if c in df_wide.columns]

df_bars = df_wide[bar_cols]
df_lines = df_wide[line_cols]

# Reverse bar order for plotting (top to bottom)
df_bars = df_bars[bar_cols[::-1]]

# Split positive and negative bars so they stack correctly around zero
df_pos = df_bars.clip(lower=0)
df_neg = df_bars.clip(upper=0)

fig, ax = plt.subplots(figsize=(28, 8))

# Positive stacked bars
df_pos.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    width=0.8,
    color=[color_map[c] for c in df_pos.columns]
)

# Negative stacked bars
df_neg.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    width=0.8,
    legend=False,
    color=[color_map[c] for c in df_neg.columns]
)

# Lines
x = np.arange(len(df_wide.index))
for col in line_cols:
    ax.plot(
        x,
        df_lines[col].values,
        linewidth=2.5,
        label=col,
        color=color_map.get(col, "black"),
        zorder=5
    )

# Formatting
ax.axhline(0, color="black", linewidth=1)
ax.grid(visible=True, axis="y")
ax.set_axisbelow(True)
ax.yaxis.set_major_locator(MultipleLocator(5))

ax.set_xlabel(None)
ax.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")
# ax.set_title("Global annual LULUCF fluxes", pad=20)

ax.set_xticks(x[::2])
ax.set_xticklabels(df_wide.index.astype(int)[::2], rotation=0)

# Build legend in desired order
handles, labels = ax.get_legend_handles_labels()
label_to_handle = dict(zip(labels, handles))

ordered_labels = (
    desired_bar_order +
    desired_line_order
)

ordered_labels = [l for l in ordered_labels if l in label_to_handle]

# Map labels to readable names
pretty_labels = [label_map.get(l, l) for l in ordered_labels]

ax.legend(
    [label_to_handle[l] for l in ordered_labels],
    pretty_labels,
    title="",
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    ncol=1,
    frameon=False
)

plt.tight_layout(rect=[0, 0, 0.80, 1])
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_global_timeseries_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# df_wide

In [ ]:
### Global LULUCF components with gross as bars and net veg and LULUCF as lines, panel by climate domain (three panels)
### Per https://chatgpt.com/g/g-p-69399a7fcc808191b337d3fac695447c-afolu-flux-model/c/69c6d457-96b4-8328-9fca-ff50f8273233
### Colors consistent with LULUCF maps from Claude (session 'LULUCF graph color consistency with maps')

df_fig = LULUCF_outputs_dropped.copy()

## Various lists and dictionaries needed to create the figure

# Drop unwanted vegetation-only emissions layers
layers_to_drop = [
    f"veg_{cn.gross_emis_all_C_pools_CO2_only_pattern}",
    f"veg_{cn.gross_emis_all_C_pools_non_CO2_only_pattern}",
]

# Net flux layers to draw as lines instead of bars
net_flux_layers = [
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
    "LULUCF_net_flux__MgCO2e",
]

# Organic soil layers to combine for graphing
organic_soil_layers = [
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]

organic_soil_combined_label = "organic_soil__all_gases__MgCO2e"

desired_bar_order = [
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2",
    "organic_soil__all_gases__MgCO2e",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}",
    f"veg_{cn.gross_removals_all_C_pools_pattern}",
]

desired_line_order = [
    "LULUCF_net_flux__MgCO2e",
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}",
]

color_map = {
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2": "#8c510a",        # emissions idx 8 (medium-dark brown)
    "organic_soil__all_gases__MgCO2e":               "#543005",         # emissions idx 9 (darkest brown)
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}": "#dfc27d",   # emissions idx 6 (medium tan)
    f"veg_{cn.gross_removals_all_C_pools_pattern}":  "#35978f",         # removals  idx 2 (medium teal)
    "LULUCF_net_flux__MgCO2e":                       "#000000",         # black line
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}": "#666633",  # green
}

label_map = {
    "SOC_change__mineral_soil_extent__0-30cm_MgCO2":
        "Mineral soil carbon change",
    "organic_soil__all_gases__MgCO2e":
        "Organic soil emissions",
    f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}":
        "Vegetation gross emissions",
    f"veg_{cn.gross_removals_all_C_pools_pattern}":
        "Vegetation gross removals",
    "LULUCF_net_flux__MgCO2e":
        "LULUCF net flux",
    # f"veg_{cn.net_flux_all_C_pools_all_gases_pattern}":
    #     "Vegetation net flux",
}

domain_order = [
    "Subtropical/tropical",
    "Temperate",
    "Boreal",
]

domain_title_map = {
    "Subtropical/tropical": "Subtropical/tropical",
    "Temperate": "Temperate",
    "Boreal": "Boreal",
}


## Creates the figure

df_fig = df_fig[~df_fig["analysis_layer"].isin(layers_to_drop)].copy()

# Keep only requested climate domains
df_fig = df_fig[df_fig["climate_domain"].isin(domain_order)].copy()

# Combine the three organic soil layers into one label
df_fig["analysis_layer"] = df_fig["analysis_layer"].replace(
    {layer: organic_soil_combined_label for layer in organic_soil_layers}
)

# Sum by climate domain, year, and analysis layer
df_ts = (
    df_fig
    .groupby(["climate_domain", "year", "analysis_layer"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)

# Convert to Gt CO2e/yr
df_ts["flux_Gt_CO2e_yr"] = df_ts["flux_Mg_CO2e_yr"] / 1e9

# Compute shared y-limits across all panels
all_panel_values = []

for domain in domain_order:
    df_domain = df_ts[df_ts["climate_domain"] == domain]

    if df_domain.empty:
        continue

    df_wide = (
        df_domain
        .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
        .fillna(0)
    )

    bar_cols = [c for c in desired_bar_order if c in df_wide.columns]
    line_cols = [c for c in desired_line_order if c in df_wide.columns]

    if bar_cols:
        df_bars = df_wide[bar_cols]
        all_panel_values.extend(df_bars.to_numpy().ravel())

    if line_cols:
        df_lines = df_wide[line_cols]
        all_panel_values.extend(df_lines.to_numpy().ravel())

y_abs_max = max(abs(np.nanmin(all_panel_values)), abs(np.nanmax(all_panel_values)))
y_pad = y_abs_max * 0.08
ymin, ymax = -y_abs_max - y_pad, y_abs_max + y_pad


## Plot

fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=True)

legend_handles = None
legend_labels = None

for ax, domain in zip(axes, domain_order):
    df_domain = df_ts[df_ts["climate_domain"] == domain]

    if df_domain.empty:
        ax.set_visible(False)
        continue

    df_wide = (
        df_domain
        .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
        .fillna(0)
    )

    # Apply ordering
    bar_cols = [c for c in desired_bar_order if c in df_wide.columns]
    line_cols = [c for c in desired_line_order if c in df_wide.columns]

    df_bars = df_wide[bar_cols]
    df_lines = df_wide[line_cols]

    # Reverse for plotting so desired_bar_order reads top-to-bottom in legend
    df_bars = df_bars[bar_cols[::-1]]

    # Split positive and negative bars
    df_pos = df_bars.clip(lower=0)
    df_neg = df_bars.clip(upper=0)

    # Plot bars
    df_pos.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        width=0.8,
        legend=False,
        color=[color_map[c] for c in df_pos.columns]
    )

    df_neg.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        width=0.8,
        legend=False,
        color=[color_map[c] for c in df_neg.columns]
    )

    # Plot lines
    x = np.arange(len(df_wide.index))
    for col in line_cols:
        ax.plot(
            x,
            df_lines[col].values,
            linewidth=2.5,
            label=col,
            color=color_map.get(col, "black"),
            zorder=5
        )

    # Formatting
    ax.axhline(0, color="black", linewidth=1)
    ax.set_axisbelow(True)
    ax.grid(visible=True, axis="y")
    ax.set_ylim(ymin, ymax)

    ax.set_title(domain_title_map.get(domain, domain), pad=12)
    ax.set_xlabel(None)

    ax.set_xticks(x[::2])
    ax.set_xticklabels(df_wide.index.astype(int)[::2], rotation=0)

    if ax is axes[0]:
        ax.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")
    else:
        ax.set_ylabel(None)

    # Save handles once for shared legend
    if legend_handles is None:
        handles, labels = ax.get_legend_handles_labels()
        label_to_handle = dict(zip(labels, handles))
        ordered_labels = [l for l in (desired_bar_order + desired_line_order) if l in label_to_handle]
        legend_handles = [label_to_handle[l] for l in ordered_labels]
        legend_labels = [label_map.get(l, l) for l in ordered_labels]

# # Shared title
# fig.suptitle("Global annual LULUCF fluxes by climate domain", y=0.98)

# Shared legend below figure
legend = fig.legend(
    legend_handles,
    legend_labels,
    title="",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=1,
    frameon=False
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.05)  # To reduce space between graphs and legend. Smaller value is smaller gap. 
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_climate_domain_timeseries_with_legend_{today}.jpg', dpi=300, bbox_inches='tight')
legend.set_visible(False)
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_climate_domain_no_legend_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

print(f"Table for {domain} (to compare against Excel spreadsheet):")
df_wide

In [ ]:
### Global flux timeseries (panel a) and climate-domain flux timeseries (panel b) for journal article
### Per Claude session 'LULUCF graph color consistency with maps'
### Claude said it was better to recreate the maps rather than trying to composite the global and climate domain images created above

# ── Figure layout ─────────────────────────────────────────────────────────────
# Panel a:  left=0.07 → right=0.70  (63% of figure width)
# Legend:   right of panel a, ending at ~0.92
# B panels: left=0.07 → right=0.92  (85% of figure width = same as a + legend)

fig = plt.figure(figsize=(20, 13))

gs_a = GridSpec(1, 1, figure=fig,
                left=0.07, right=0.70, top=0.93, bottom=0.52)
ax_a = fig.add_subplot(gs_a[0, 0])

gs_b = GridSpec(1, 3, figure=fig,
                left=0.07, right=0.92, top=0.45, bottom=0.08,
                wspace=0.25)
ax_b0 = fig.add_subplot(gs_b[0, 0])
ax_b1 = fig.add_subplot(gs_b[0, 1], sharey=ax_b0)
ax_b2 = fig.add_subplot(gs_b[0, 2], sharey=ax_b0)
axes_b = [ax_b0, ax_b1, ax_b2]

# ── Panel a: global ───────────────────────────────────────────────────────────

df_fig_a = LULUCF_outputs_dropped.copy()
df_fig_a = df_fig_a[~df_fig_a["analysis_layer"].isin(layers_to_drop)].copy()
df_fig_a["analysis_layer"] = df_fig_a["analysis_layer"].replace(
    {layer: organic_soil_combined_label for layer in organic_soil_layers}
)
df_ts_a = (
    df_fig_a
    .groupby(["year", "analysis_layer"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)
df_ts_a["flux_Gt_CO2e_yr"] = df_ts_a["flux_Mg_CO2e_yr"] / 1e9
df_wide_a = (
    df_ts_a
    .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
    .fillna(0)
)

bar_cols_a  = [c for c in desired_bar_order  if c in df_wide_a.columns]
line_cols_a = [c for c in desired_line_order if c in df_wide_a.columns]
df_bars_a   = df_wide_a[bar_cols_a[::-1]]

df_bars_a.clip(lower=0).plot(kind="bar", stacked=True, ax=ax_a, width=0.8, legend=False,
                              color=[color_map[c] for c in df_bars_a.columns])
df_bars_a.clip(upper=0).plot(kind="bar", stacked=True, ax=ax_a, width=0.8, legend=False,
                              color=[color_map[c] for c in df_bars_a.columns])

x_a = np.arange(len(df_wide_a.index))
for col in line_cols_a:
    ax_a.plot(x_a, df_wide_a[col].values, linewidth=2.5,
              color=color_map.get(col, "black"), zorder=5)

ax_a.axhline(0, color="black", linewidth=1)
ax_a.grid(visible=True, axis="y")
ax_a.set_axisbelow(True)
ax_a.yaxis.set_major_locator(MultipleLocator(5))
ax_a.set_xlabel(None)
ax_a.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")
ax_a.set_xticks(x_a[::2])
ax_a.set_xticklabels(df_wide_a.index.astype(int)[::2], rotation=0)

# ── Panel b: climate domains ──────────────────────────────────────────────────

df_fig_b = LULUCF_outputs_dropped.copy()
df_fig_b = df_fig_b[~df_fig_b["analysis_layer"].isin(layers_to_drop)].copy()
df_fig_b = df_fig_b[df_fig_b["climate_domain"].isin(domain_order)].copy()
df_fig_b["analysis_layer"] = df_fig_b["analysis_layer"].replace(
    {layer: organic_soil_combined_label for layer in organic_soil_layers}
)
df_ts_b = (
    df_fig_b
    .groupby(["climate_domain", "year", "analysis_layer"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)
df_ts_b["flux_Gt_CO2e_yr"] = df_ts_b["flux_Mg_CO2e_yr"] / 1e9

# Shared y-limits across b panels
all_b_vals = []
for domain in domain_order:
    df_d = df_ts_b[df_ts_b["climate_domain"] == domain]
    if not df_d.empty:
        df_w = df_d.pivot(index="year", columns="analysis_layer",
                          values="flux_Gt_CO2e_yr").fillna(0)
        all_b_vals.extend(df_w.to_numpy().ravel())

y_abs_max_b = max(abs(np.nanmin(all_b_vals)), abs(np.nanmax(all_b_vals)))
y_pad_b = y_abs_max_b * 0.08

for ax, domain in zip(axes_b, domain_order):
    df_domain = df_ts_b[df_ts_b["climate_domain"] == domain]
    if df_domain.empty:
        ax.set_visible(False)
        continue

    df_wide = (
        df_domain
        .pivot(index="year", columns="analysis_layer", values="flux_Gt_CO2e_yr")
        .fillna(0)
    )
    bar_cols  = [c for c in desired_bar_order  if c in df_wide.columns]
    line_cols = [c for c in desired_line_order if c in df_wide.columns]
    df_bars   = df_wide[bar_cols[::-1]]

    df_bars.clip(lower=0).plot(kind="bar", stacked=True, ax=ax, width=0.8, legend=False,
                                color=[color_map[c] for c in df_bars.columns])
    df_bars.clip(upper=0).plot(kind="bar", stacked=True, ax=ax, width=0.8, legend=False,
                                color=[color_map[c] for c in df_bars.columns])

    x = np.arange(len(df_wide.index))
    for col in line_cols:
        ax.plot(x, df_wide[col].values, linewidth=2.5,
                color=color_map.get(col, "black"), zorder=5)

    ax.axhline(0, color="black", linewidth=1)
    ax.set_axisbelow(True)
    ax.grid(visible=True, axis="y")
    ax.set_ylim(-y_abs_max_b - y_pad_b, y_abs_max_b + y_pad_b)
    ax.set_title(domain_title_map.get(domain, domain), pad=12)
    ax.set_xlabel(None)
    ax.set_xticks(x[::2])
    ax.set_xticklabels(df_wide.index.astype(int)[::2], rotation=0)
    if ax is axes_b[0]:
        ax.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")

# ── Panel labels ──────────────────────────────────────────────────────────────

ax_a.text(-0.02, 1.03, "a", transform=ax_a.transAxes,
          fontsize=14, fontweight="bold", va="bottom")
axes_b[0].text(-0.10, 1.03, "b", transform=axes_b[0].transAxes,
               fontsize=14, fontweight="bold", va="bottom")

# ── Legend ────────────────────────────────────────────────────────────────────

legend_items = [
    Patch(facecolor=color_map["SOC_change__mineral_soil_extent__0-30cm_MgCO2"],
          label=label_map["SOC_change__mineral_soil_extent__0-30cm_MgCO2"]),
    Patch(facecolor=color_map["organic_soil__all_gases__MgCO2e"],
          label=label_map["organic_soil__all_gases__MgCO2e"]),
    Patch(facecolor=color_map[f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"],
          label=label_map[f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"]),
    Patch(facecolor=color_map[f"veg_{cn.gross_removals_all_C_pools_pattern}"],
          label=label_map[f"veg_{cn.gross_removals_all_C_pools_pattern}"]),
    Line2D([0], [0], color=color_map["LULUCF_net_flux__MgCO2e"], linewidth=2.5,
           label=label_map["LULUCF_net_flux__MgCO2e"]),
]

# Anchor legend just right of panel a, vertically centred with it
fig.canvas.draw()
ax_a_pos  = ax_a.get_position()
legend_x  = ax_a_pos.x1 + 0.01
legend_y  = ax_a_pos.y0 + ax_a_pos.height / 2

legend = fig.legend(
    handles=legend_items,
    loc="center left",
    bbox_to_anchor=(legend_x, legend_y),
    frameon=False,
)

# ── Save ──────────────────────────────────────────────────────────────────────

# fig.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_global_and_climate_domain_with_legend_{today}.jpg', dpi=300, bbox_inches='tight')

In [ ]:
### LULUCF waterfall diagram
### Made with Claude ('Create LULUCF waterfall chart')

# ── 0. Aggregate raw data into summary rows ──────────────────────────────────
raw = LULUCF_outputs_dropped[
    ~LULUCF_outputs_dropped["analysis_layer"].str.contains("gross_emissions|gross_removals")
].copy()

n_years = raw["year"].nunique()

def annual_Gt(df, mask):
    """Sum all pixel-years matching mask, divide by n_years → annual average in Gt."""
    return df.loc[mask, "flux_Mg_CO2e_yr"].sum() / n_years / 1e9

rows_summary = []
def add_row(section, subsection, category, val):
    rows_summary.append({
        "Section": section, "Subsection": subsection,
        "Category": category, "All_gases_Gt_CO2e_yr": val,
    })

veg  = raw["LULUCF_component"] == "vegetation"
tree = raw["land_state_broad_class"] == "tree"
dlc  = raw["land_state_detailed_class"]

add_row("Vegetation", "Tall vegetation", "New tree cover",
        annual_Gt(raw, veg & tree & (dlc == "tree_gain")))
add_row("Vegetation", "Tall vegetation", "Undisturbed tree cover (including fires)",
        annual_Gt(raw, veg & tree & (dlc == "tree_tree_undisturbed")))
add_row("Vegetation", "Tall vegetation", "Partially disturbed tree cover",
        annual_Gt(raw, veg & tree & (dlc.isin(["tree_tree_disturbed", "tree_tree_disturbed_fire_only"]))))
add_row("Vegetation", "Tall vegetation", "Tree cover loss",
        annual_Gt(raw, veg & tree & (dlc == "tree_loss")))
add_row("Vegetation", "Tall vegetation", "Total",
        annual_Gt(raw, veg & tree))
add_row("Vegetation", "Non-tall vegetation", "All classes",
        annual_Gt(raw, veg & (raw["tall_veg_type"] == "non_tall_vegetation")))
add_row("Vegetation", "All vegetation", "Total",
        annual_Gt(raw, veg))
add_row("Soil", "Organic soil", "Fire",
        annual_Gt(raw, raw["analysis_layer"].str.contains("organic_soil_fire")))
add_row("Soil", "Organic soil", "Extraction",
        annual_Gt(raw, raw["analysis_layer"].str.contains("organic_soil_extraction")))
add_row("Soil", "Organic soil", "Drainage",
        annual_Gt(raw, raw["analysis_layer"].str.contains("organic_soil_drainage")))
add_row("Soil", "Organic soil", "Total",
        annual_Gt(raw, raw["LULUCF_component"] == "organic_soil"))
add_row("Soil", "Mineral soil", "All classes",
        annual_Gt(raw, raw["LULUCF_component"] == "mineral_soil"))
add_row("Soil", "All soil", "Total",
        annual_Gt(raw, raw["LULUCF_component"].isin(["organic_soil", "mineral_soil"])))
add_row("LULUCF", "All components", "Total",
        annual_Gt(raw, raw["analysis_layer"] == "LULUCF_net_flux__MgCO2e"))

df_wf = pd.DataFrame(rows_summary)

veg_total    = df_wf.loc[df_wf["Subsection"] == "All vegetation", "All_gases_Gt_CO2e_yr"].values[0]
soil_total   = df_wf.loc[df_wf["Subsection"] == "All soil",       "All_gases_Gt_CO2e_yr"].values[0]
lulucf_total = df_wf.loc[df_wf["Section"]    == "LULUCF",         "All_gases_Gt_CO2e_yr"].values[0]

def section_subtitle(val):
    term = "net sink" if val < 0 else "net source"
    return f"({term} = {val:.3g} Gt CO₂e yr⁻¹)"

section_subtitle_map = {
    "Vegetation": section_subtitle(veg_total),
    "Soil":       section_subtitle(soil_total),
    "LULUCF":     section_subtitle(lulucf_total),
}

# ── 1. Select detail rows + LULUCF total as closing bar ─────────────────────
detail_mask = (
    ((df_wf["Subsection"] == "Tall vegetation")   & (df_wf["Category"] != "Total")) |
    ( df_wf["Subsection"] == "Non-tall vegetation")                                  |
    ((df_wf["Subsection"] == "Organic soil")       & (df_wf["Category"] != "Total")) |
    ( df_wf["Subsection"] == "Mineral soil")                                         |
    ((df_wf["Section"]    == "LULUCF")             & (df_wf["Category"] == "Total"))
)
wf = df_wf[detail_mask][["Section", "Subsection", "Category", "All_gases_Gt_CO2e_yr"]].reset_index(drop=True)
wf["is_total"] = (wf["Section"] == "LULUCF") & (wf["Category"] == "Total")

# ── 2. Waterfall geometry ────────────────────────────────────────────────────
vals    = wf["All_gases_Gt_CO2e_yr"].values.astype(float)
bottoms = np.zeros(len(vals))
running = 0.0

for i in range(len(vals)):
    if wf["is_total"].iloc[i]:
        bottoms[i] = 0.0
    else:
        bottoms[i] = running
        running   += vals[i]

# ── 3. Colors ────────────────────────────────────────────────────────────────
SINK   = "#33cc33"
SOURCE = "#d279d2"
TOTAL  = "#616161"

colors = [
    TOTAL  if wf["is_total"].iloc[i] else
    SINK   if vals[i] < 0            else
    SOURCE
    for i in range(len(vals))
]

# ── 4. Tick labels ───────────────────────────────────────────────────────────
def make_label(row):
    if row["is_total"]:
        return "LULUCF\nTotal"
    cat, sub = row["Category"], row["Subsection"]
    short = {
        "Undisturbed tree cover (including fires)": "Undisturbed\ntree cover",
        "Partially disturbed tree cover":           "Partially\ndisturbed\ntree cover",
        "New tree cover":                           "New\ntree cover",
        "Tree cover loss":                          "Tree cover\nloss",
        "Fire":                                     "Organic soil\nFire",
        "Extraction":                               "Organic soil\nExtraction",
        "Drainage":                                 "Organic soil\nDrainage",
    }
    if cat == "All classes":
        return "Short veg. &\ncropland" if sub == "Non-tall vegetation" else sub
    return short.get(cat, cat)

labels = [make_label(row) for _, row in wf.iterrows()]

# ── 5. Section midpoints for top axis ───────────────────────────────────────
sections = wf["Section"].values
section_groups = {}
for i, s in enumerate(sections):
    section_groups.setdefault(s, []).append(i)

# ── 6. Plot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))

bar_bottoms = np.where(vals < 0, bottoms + vals, bottoms)
bar_heights = np.abs(vals)

bars = ax.bar(
    range(len(vals)),
    bar_heights,
    bottom=bar_bottoms,
    color=colors,
    edgecolor="white",
    linewidth=0.8,
    width=0.65,
)

ax.grid(False)
ax.yaxis.grid(True)
ax.set_axisbelow(True)

# Dashed horizontal connectors between consecutive flow bars
for i in range(len(vals) - 1):
    if not wf["is_total"].iloc[i] and not wf["is_total"].iloc[i + 1]:
        y = bottoms[i] + vals[i]
        ax.plot([i + 0.33, i + 0.67], [y, y],
                color="grey", linewidth=0.8, linestyle="--", alpha=0.6)

# Bar value labels
SMALL_BAR = 0.6
for i, (bar, val) in enumerate(zip(bars, vals)):
    label_text = f"{val:+.2g}"
    if bar.get_height() < SMALL_BAR:
        if val >= 0:
            y_pos, va = bar.get_y() + bar.get_height() + 0.15, "bottom"
        else:
            y_pos, va = bar.get_y() - 0.15, "top"
        ax.text(bar.get_x() + bar.get_width() / 2, y_pos, label_text,
                ha="center", va=va, fontsize=12, color="black", fontweight="bold")
    else:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_y() + bar.get_height() / 2, label_text,
                ha="center", va="center", fontsize=12, color="white", fontweight="bold")

# Zero reference line
ax.axhline(0, color="black", linewidth=0.9)

# Solid vertical section dividers
for i in range(1, len(sections)):
    if sections[i] != sections[i - 1]:
        ax.axvline(i - 0.5, color="#555555", linewidth=1.2, linestyle="-", alpha=0.35)

# Bottom axis
ax.set_xticks(range(len(vals)))
ax.set_xticklabels(labels, rotation=0, ha="center")
ax.tick_params(axis="x", labelsize=10)
ax.set_title("LULUCF GHG flux components (all GHGs, average for 2016-2024)", pad=20)

# Y-axis: no tick labels, 2.5 Gt gridlines, scale bar annotation
ax.yaxis.set_major_locator(MultipleLocator(2.5))
ax.set_yticklabels([])
ax.set_ylabel("")

trans = blended_transform_factory(ax.transAxes, ax.transData)
ax.annotate("", xy=(-0.03, 0), xytext=(-0.03, -2.5),
            xycoords=trans, textcoords=trans,
            annotation_clip=False,
            arrowprops=dict(arrowstyle="<->", color="black", lw=1.0))
txt = ax.text(-0.05, -1.25, "2.5 Gt\nCO₂e yr⁻¹",
              transform=trans, ha="right", va="center", fontsize=9)
txt.set_clip_on(False)

# Top axis: section names (bold, full size) + smaller subtitles as separate text
ax2 = ax.twiny()
ax2.set_xlim(ax.get_xlim())
ax2.set_xticks([np.mean(idxs) for idxs in section_groups.values()])
ax2.set_xticklabels(list(section_groups.keys()), fontweight="bold")
ax2.tick_params(length=0, pad=18)   # pad lifts names up, leaving room for subtitle
ax2.grid(False)

# Subtitles slotted between top spine and section names
trans2 = blended_transform_factory(ax2.transData, ax2.transAxes)
for section, idxs in section_groups.items():
    t = ax2.text(
        np.mean(idxs), 1.01,
        section_subtitle_map[section],
        transform=trans2,
        ha="center", va="bottom",
        fontsize=9,
        clip_on=False,
    )

# Hide both top spines to remove double line
ax.spines["top"].set_visible(False)
ax2.spines["bottom"].set_visible(False)

# Legend
ax.legend(
    handles=[
        mpatches.Patch(color=SINK,   label="Sink (−)"),
        mpatches.Patch(color=SOURCE, label="Source (+)"),
        mpatches.Patch(color=TOTAL,  label="Net sink"),
    ],
    loc="lower right", fontsize=9
)

plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_global_waterfall_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()
df_wf

In [ ]:
### Global vegetation gross emissions timeseries by fire vs. non-fire, with and without Australia (single panel)
### Per Claude ("Create global emissions timeseries graph")

# ---- Setup ----
context_to_analyze = "land_state_node"

df_fig = LULUCF_outputs_dropped.copy()

# Filter to gross emissions (all gases)
df_emis = df_fig[df_fig["analysis_layer"] == f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"].copy()

# ---- Reclassify fire vs non-fire BEFORE aggregation ----
df_emis["fire_class"] = df_emis[context_to_analyze].apply(
    lambda x: "Fire" if re.search(r"9(0*)$", str(int(float(x)))) else "Non-fire"
)

# ---- Aggregate WITH Australia ----
df_with_Australia = (
    df_emis
    .groupby(["fire_class", "year"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)
df_with_Australia["Australia_group"] = "with Australia"
df_with_Australia

# ---- Aggregate WITHOUT Australia ----
df_without_Australia = (
    df_emis[df_emis["country_name"] != "Australia"]
    .groupby(["fire_class", "year"], as_index=False)
    .agg({"flux_Mg_CO2e_yr": "sum"})
)
df_without_Australia["Australia_group"] = "without Australia"

# Combine
df_emis_ts = pd.concat([df_with_Australia, df_without_Australia], ignore_index=True)

# Convert to Gt
df_emis_ts["value_Gt"] = df_emis_ts["flux_Mg_CO2e_yr"] / 1e9

# Create combined plotting label
df_emis_ts["series"] = df_emis_ts["fire_class"] + " (" + df_emis_ts["Australia_group"] + ")"
df_emis_ts

# ---- Plot ----
series_order = [
    "Non-fire (with Australia)",
    "Non-fire (without Australia)",
    "Fire (with Australia)",
    "Fire (without Australia)",
]

series_palette = {
    "Non-fire (with Australia)": "#1f77b4",
    "Non-fire (without Australia)": "#1f77b4",
    "Fire (with Australia)":     "#d62728",
    "Fire (without Australia)":  "#d62728",
}

series_dashes = {
    "Non-fire (with Australia)": "",
    "Non-fire (without Australia)": (5, 2),
    "Fire (with Australia)":     "",
    "Fire (without Australia)":  (5, 2),
}

plt.figure(figsize=(10, 5))

ax = sns.lineplot(
    data=df_emis_ts,
    x="year",
    y="value_Gt",
    hue="series",
    style="series",
    hue_order=series_order,
    style_order=series_order,
    palette=series_palette,
    dashes=series_dashes,
    linewidth=2.5
)

ax.axhline(0, color="black", linewidth=1)
ax.grid(False)

ax.set_xlabel(None)
ax.set_ylabel("Gt CO$_2$e yr$^{-1}$")
# ax.set_title("Gross emissions (all gases): fire vs non-fire, with and without Australia", pad=20)

ax.legend(
    title="Emission type",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/veg_emis_timeseries_fire_Australia_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### Bar graphs of countries with largest net sources and sinks by continent
### Country-level gross/net LULUCF fluxes by region (horizontal stacked bars, net as dot)
### Top 3 net emitters + top 3 net removers per region + Other countries
### From Claude ('Bar graph of countries with largest net sources-sinks')

df_fig = LULUCF_outputs_dropped.copy()

n_years = len(cn.interval_end_years_annual)

# ── Layer definitions ───────────────────────────────────────────────────────

VEG_EMIS  = f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"
VEG_REMOV = f"veg_{cn.gross_removals_all_C_pools_pattern}"
NET_LAYER = "LULUCF_net_flux__MgCO2e"

ORGANIC_SOIL_LAYERS = [
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]
MINERAL_SOIL_LAYER = "SOC_change__mineral_soil_extent__0-30cm_MgCO2"
ORGANIC_LABEL = "organic_soil__all_gases__MgCO2e"

df_fig["analysis_layer"] = df_fig["analysis_layer"].replace(
    {l: ORGANIC_LABEL for l in ORGANIC_SOIL_LAYERS}
)

VEG_NET_LAYER = "veg_net_flux__all_C_pools__all_gases__MgCO2e"
LAYERS_NEEDED = [VEG_EMIS, VEG_REMOV, MINERAL_SOIL_LAYER, ORGANIC_LABEL, NET_LAYER, VEG_NET_LAYER]
df_fig = df_fig[df_fig["analysis_layer"].isin(LAYERS_NEEDED)].copy()

# ── Annual average flux per country × analysis layer ────────────────────────

df_country = (
    df_fig
    .groupby(["region_L1", "country_name", "analysis_layer"], as_index=False)
    .agg(flux_Mg=("flux_Mg_CO2e_yr", "sum"))
)
df_country["flux_Mt"] = df_country["flux_Mg"] / n_years / 1e6

df_wide = df_country.pivot_table(
    index=["region_L1", "country_name"],
    columns="analysis_layer",
    values="flux_Mt",
    aggfunc="sum",
    fill_value=0,
).reset_index()
df_wide.columns.name = None

for col in [VEG_EMIS, VEG_REMOV, MINERAL_SOIL_LAYER, ORGANIC_LABEL, NET_LAYER, VEG_NET_LAYER]:
    if col not in df_wide.columns:
        df_wide[col] = 0.0

df_wide["gross_emis_Mt"]  = df_wide[VEG_EMIS] + df_wide[ORGANIC_LABEL] + df_wide[MINERAL_SOIL_LAYER]
df_wide["gross_remov_Mt"] = df_wide[VEG_REMOV]
df_wide["net_Mt"]         = df_wide[NET_LAYER]
df_wide["veg_net_Mt"]     = df_wide[VEG_NET_LAYER]

# ── Drop Antarctica and unassigned regions ────────────────────────────────────

REGIONS_TO_DROP = {"Antarctica", "Unassigned"}
df_wide = df_wide[~df_wide["region_L1"].isin(REGIONS_TO_DROP)].copy()

# ── Country selection ─────────────────────────────────────────────────────────

def select_countries(group, region_name):
    top_emis  = group.nlargest(3,  "net_Mt")["country_name"].tolist()
    top_remov = group.nsmallest(3, "net_Mt")["country_name"].tolist()
    featured  = list(dict.fromkeys(top_emis + top_remov))

    rows = []
    for cname in featured:
        r = group[group["country_name"] == cname].iloc[0].copy()
        r["region_L1"] = region_name
        r["featured_group"] = "source" if cname in top_emis else "sink"
        rows.append(r)

    other_mask = ~group["country_name"].isin(featured)
    if other_mask.any():
        other = group[other_mask].sum(numeric_only=True)
        other["country_name"] = "Other countries"
        other["region_L1"]       = region_name
        other["featured_group"] = "other"
        rows.append(other)

    return pd.DataFrame(rows)

df_plot = (
    df_wide
    .groupby("region_L1", group_keys=False)
    .apply(lambda grp: select_countries(grp, grp.name), include_groups=False)
    .reset_index(drop=True)
)

# ── Colour / style constants ──────────────────────────────────────────────────

COLOR_VEG_EMIS  = "#4d4d8f"
COLOR_ORG_EMIS  = "#7b7bbf"
COLOR_MIN_EMIS  = "#b3b3d9"
COLOR_VEG_REMOV = "#2d7d2d"
DOT_COLOR       = "black"
DOT_SIZE        = 21
VEG_DOT_SIZE    = DOT_SIZE * 0.7
X_TICKS         = [-3000, -2000, -1000, 0, 1000, 2000, 3000]

LEGEND_ORDER = [
    "Veg net flux",
    "LULUCF net flux",
    "Veg gross removals",
    "Veg gross emissions",
    "Organic soil emissions",
    "Mineral soil emissions",
]

# ── Build the figure ──────────────────────────────────────────────────────────

regions = sorted(df_plot["region_L1"].unique())
n_regions = len(regions)
ncols = 2
nrows = (n_regions + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3.5), squeeze=False)

x_abs_max = max(abs(v) for v in X_TICKS)
x_pad = x_abs_max * 0.06
xlim = (-x_abs_max - x_pad, x_abs_max + x_pad)

_row_mids = {}

for idx, region in enumerate(regions):
    row = idx // ncols
    col = idx % ncols
    ax  = axes[row][col]
    is_bottom      = (row == nrows - 1)
    is_right_panel = (col == 1)

    df_r = df_plot[df_plot["region_L1"] == region].copy()
    other_row  = df_r[df_r["featured_group"] == "other"]
    emis_r     = df_r[df_r["featured_group"] == "source"].sort_values("net_Mt", ascending=False)
    remov_r    = df_r[df_r["featured_group"] == "sink"].sort_values("net_Mt", ascending=False)
    feature_r  = pd.concat([emis_r, remov_r], ignore_index=True)
    df_r       = pd.concat([other_row, feature_r], ignore_index=True)

    countries = df_r["country_name"].tolist()
    y = np.arange(len(countries))

    veg_e = df_r[VEG_EMIS].values
    org_e = df_r[ORGANIC_LABEL].values
    min_e = df_r[MINERAL_SOIL_LAYER].values

    ax.barh(y, veg_e,                   height=0.6, color=COLOR_VEG_EMIS,  label="Veg gross emissions")
    ax.barh(y, org_e, left=veg_e,       height=0.6, color=COLOR_ORG_EMIS,  label="Organic soil emissions")
    ax.barh(y, min_e, left=veg_e+org_e, height=0.6, color=COLOR_MIN_EMIS,  label="Mineral soil emissions")
    ax.barh(y, df_r["gross_remov_Mt"].values, height=0.6,
            color=COLOR_VEG_REMOV, label="Veg gross removals")
    ax.scatter(df_r["net_Mt"].values, y, color=DOT_COLOR, zorder=5, s=DOT_SIZE, label="LULUCF net flux")
    ax.scatter(df_r["veg_net_Mt"].values, y, color="#888888", zorder=5, s=VEG_DOT_SIZE, label="Veg net flux", marker="o")

    # Dashed lines: between source/sink groups, and between Other and sources
    n_emis = len(emis_r)
    if n_emis > 0 and len(remov_r) > 0:
        ax.axhline(y=n_emis + 0.5, color="gray", linestyle="--", linewidth=0.8, zorder=3)
    if n_emis > 0:
        ax.axhline(y=0.5, color="gray", linestyle="--", linewidth=0.8, zorder=3)
    if col == 0:
        _row_mids[row] = (
            (1 + n_emis) / 2 if n_emis > 0 else None,
            n_emis + 0.5 + len(remov_r) / 2 if len(remov_r) > 0 else None,
            ax,
        )

    ax.set_yticks(y)
    ax.set_yticklabels([])
    ax.tick_params(axis="y", left=False, right=False)
    x_label = xlim[0] + (xlim[1] - xlim[0]) * 0.01
    for yi, name in zip(y, countries):
        ax.text(x_label, yi, name, ha="left", va="center", fontsize=8, clip_on=True)
    ax.set_xlim(xlim)
    ax.set_xticks(X_TICKS)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(region, fontsize=10, fontweight="bold")
    ax.grid(axis="x", linewidth=0.5, alpha=0.5, which="major")
    ax.set_axisbelow(True)

    if is_bottom:
        ax.set_xlabel("Average annual flux (MtCO₂e yr⁻¹)", fontsize=7)
        ax.tick_params(axis="x", labelsize=7)
    else:
        ax.set_xlabel(None)
        ax.tick_params(axis="x", labelbottom=False)

for idx in range(n_regions, nrows * ncols):
    axes[idx // ncols][idx % ncols].set_visible(False)

# ── Legend in specified order ─────────────────────────────────────────────────

handles, labels = axes[0][0].get_legend_handles_labels()
label_to_handle = dict(zip(labels, handles))
ordered_handles = [label_to_handle[l] for l in LEGEND_ORDER if l in label_to_handle]
ordered_labels  = [l for l in LEGEND_ORDER if l in label_to_handle]

fig.legend(
    ordered_handles, ordered_labels,
    loc="lower center",
    ncol=5,
    frameon=False,
    fontsize=12,
    bbox_to_anchor=(0.5, -0.04),
)

# fig.suptitle("Average annual LULUCF fluxes by region and country", fontsize=13, y=1.01)
plt.tight_layout()

for row_idx, (y_src, y_snk, ax_left) in _row_mids.items():
    if not axes[row_idx][1].get_visible():
        continue
    pos_l = ax_left.get_position()
    pos_r = axes[row_idx][1].get_position()
    x_mid = (pos_l.x1 + pos_r.x0) / 2
    for y_dat, lbl in [(y_src, "Largest sources"), (y_snk, "Largest sinks")]:
        if y_dat is None:
            continue
        _, y_fig = fig.transFigure.inverted().transform(
            ax_left.transData.transform([0, y_dat])
        )
        fig.text(x_mid, y_fig, lbl, ha="center", va="center",
                 fontsize=7, color="dimgray", style="italic", rotation=90)

# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_region_country_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# # To get stats for checking
# print(LULUCF_outputs_dropped['analysis_layer'].unique())
#
# print(LULUCF_outputs_dropped[
#     (LULUCF_outputs_dropped["adm0"] == "AUS") &
#     (LULUCF_outputs_dropped["analysis_layer"] == "organic_soil_drainage__all_gases__MgCO2e")
# ]["flux_Mg_CO2e_yr"].sum()/9/10**9)
#
# df_wide[df_wide["region_L1"] == "South America"][["country_name", "net_Mt"]].sort_values("net_Mt")

In [ ]:
### Pan et al. 2024-style regional map: average annual LULUCF component fluxes by UN geoscheme region
### From Claude ('Pan et al. regional LULUCF flux map')

VEG_GROSS_EMIS    = f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"
VEG_GROSS_REMOV   = f"veg_{cn.gross_removals_all_C_pools_pattern}"
VEG_NET_LAYER     = "veg_net_flux__all_C_pools__all_gases__MgCO2e"
MINERAL_SOIL_LAYER = "SOC_change__mineral_soil_extent__0-30cm_MgCO2"
ORGANIC_SOIL_LAYERS = [
    "organic_soil_drainage__all_gases__MgCO2e",
    "organic_soil_extraction__all_gases__MgCO2e",
    "organic_soil_fire__all_gases__MgCO2e",
]
NET_LULUCF_LAYER = "LULUCF_net_flux__MgCO2e"

ALL_LAYERS = (
    [VEG_GROSS_EMIS, VEG_GROSS_REMOV, VEG_NET_LAYER, MINERAL_SOIL_LAYER]
    + ORGANIC_SOIL_LAYERS + [NET_LULUCF_LAYER]
)
LAYER_TO_COMP = {
    VEG_GROSS_EMIS:     "veg_gross_emis",
    VEG_GROSS_REMOV:    "veg_gross_remov",
    VEG_NET_LAYER:      "veg_net",
    MINERAL_SOIL_LAYER: "mineral_soil",
    **{l: "organic_soil" for l in ORGANIC_SOIL_LAYERS},
    NET_LULUCF_LAYER:   "total_LULUCF",
}

df_map = LULUCF_outputs_dropped[LULUCF_outputs_dropped["analysis_layer"].isin(ALL_LAYERS)].copy()
df_map["plot_component"] = df_map["analysis_layer"].map(LAYER_TO_COMP)
df_map["map_region"] = np.where(
    df_map["region_L1"] == "Oceania", "Oceania", df_map["region_L2_L3"]
)
df_map = df_map[
    df_map["map_region"].notna() & ~df_map["map_region"].isin(["Antarctica", "Unassigned"])
]
regional_avg = (
    df_map
    .groupby(["map_region", "plot_component", "year"])["flux_Mg_CO2e_yr"]
    .sum()
    .groupby(level=["map_region", "plot_component"])
    .mean()
    .reset_index(name="avg_flux")
)
regional_wide = regional_avg.pivot(
    index="map_region", columns="plot_component", values="avg_flux"
)

ROBINSON_CRS = cn.Robinson_crs

def _iso_to_map_region(iso):
    if not isinstance(iso, str) or iso == "-99":
        return None
    if cn.iso_to_region_UN_geoscheme_L1.get(iso) == "Oceania":
        return "Oceania"
    return cn.iso_to_region_UN_geoscheme_L2_L3.get(iso)

try:
    world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))
except AttributeError:
    _ne_url = "https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip"
    world = gpd.read_file(_ne_url)
    world = world.rename(columns={"ISO_A3_EH": "iso_a3"})

world = world[world["iso_a3"] != "ATA"].copy()
world["map_region"] = world["iso_a3"].map(_iso_to_map_region)

all_regions_sorted = sorted([
    "Northern Africa", "Western Africa", "Middle Africa", "Eastern Africa", "Southern Africa",
    "Northern Europe", "Western Europe", "Southern Europe", "Eastern Europe",
    "Western Asia", "Central Asia", "Southern Asia", "Eastern Asia", "South-eastern Asia",
    "Northern America", "Central America", "Caribbean", "South America", "Oceania",
])
_cmap = plt.colormaps["tab20"].resampled(len(all_regions_sorted))
REGION_COLORS = {region: mcolors.to_hex(_cmap(i)) for i, region in enumerate(all_regions_sorted)}

world["map_color"] = world["map_region"].map(lambda r: REGION_COLORS.get(r, "#d9d9d9"))
world = world.to_crs(ROBINSON_CRS)

BAR_POSITIONS_WGS84 = {
    "Northern Africa":    ( 15,  27),
    "Western Africa":     (-12,  10),
    "Middle Africa":      ( 22,  -3),
    "Eastern Africa":     ( 40,   2),
    "Southern Africa":    ( 27, -27),
    "Northern Europe":    ( 18,  68),
    "Western Europe":     (-12,  48),
    "Southern Europe":    ( 18,  40),
    "Eastern Europe":     ( 38,  55),
    "Western Asia":       ( 45,  24),
    "Central Asia":       ( 68,  44),
    "Southern Asia":      ( 82,  22),
    "Eastern Asia":       (105,  35),
    "South-eastern Asia": (120,   5),
    "Northern America":   (-100,  56),
    "Central America":    ( -92,  14),
    "Caribbean":          ( -60,  20),
    "South America":      ( -62, -12),
    "Oceania":            ( 152, -25),
}

_anchor_gdf = gpd.GeoDataFrame(
    {"region": list(BAR_POSITIONS_WGS84.keys())},
    geometry=[Point(lon, lat) for lon, lat in BAR_POSITIONS_WGS84.values()],
    crs="EPSG:4326",
).to_crs(ROBINSON_CRS)

BAR_POSITIONS = {
    row["region"]: (row.geometry.x, row.geometry.y)
    for _, row in _anchor_gdf.iterrows()
}

# Change 3: match gross veg colors to timeseries color_map (cell 35)
COMP_COLORS = {
    "veg_gross_emis":  "#d279d2",   # light purple-pink, same as timeseries
    "veg_gross_remov": "#33cc33",   # bright green, same as timeseries
    "organic_soil":    "#993399",
    "mineral_soil":    "#8c564b",
}
VEG_NET_EDGE_COLOR = "#1a6b3a"

# Change 4: narrow bars by 20% (250_000 → 200_000)
SUB_HW   = 200_000
HALF_GAP = 50_000

comp_cols_for_scale = ["veg_gross_emis", "veg_gross_remov", "organic_soil", "mineral_soil"]
max_abs = regional_wide[comp_cols_for_scale].abs().max().max()
SCALE   = 3_000_000 / max_abs

fig, ax = plt.subplots(figsize=(22, 11))
ax.set_aspect("equal")
ax.set_axis_off()

bounds = world.total_bounds
ax.set_xlim(bounds[0], bounds[2])
ax.set_ylim(bounds[1], bounds[3])

world.plot(ax=ax, color=world["map_color"].tolist(), edgecolor="white", linewidth=0.3, zorder=1)

for region, row in regional_wide.iterrows():
    if region not in BAR_POSITIONS:
        continue
    cx, cy = BAR_POSITIONS[region]
    cx_veg  = cx - SUB_HW - HALF_GAP
    cx_soil = cx + SUB_HW + HALF_GAP
    veg_left   = cx_veg  - SUB_HW
    veg_right  = cx_veg  + SUB_HW
    soil_left  = cx_soil - SUB_HW
    soil_right = cx_soil + SUB_HW

    veg_pos_top = veg_neg_bot = 0.0
    for comp in ["veg_gross_emis", "veg_gross_remov"]:
        val = row.get(comp, np.nan)
        if pd.isna(val): continue
        h = val * SCALE
        bottom = veg_pos_top if val >= 0 else veg_neg_bot
        if val >= 0: veg_pos_top += h
        else: veg_neg_bot += h
        ax.add_patch(mpatches.Rectangle(
            (veg_left, cy + bottom), SUB_HW * 2, h,
            facecolor=COMP_COLORS[comp], edgecolor="none", zorder=4,
        ))

    soil_pos_top = soil_neg_bot = 0.0
    for comp in ["organic_soil", "mineral_soil"]:
        val = row.get(comp, np.nan)
        if pd.isna(val): continue
        h = val * SCALE
        bottom = soil_pos_top if val >= 0 else soil_neg_bot
        if val >= 0: soil_pos_top += h
        else: soil_neg_bot += h
        ax.add_patch(mpatches.Rectangle(
            (soil_left, cy + bottom), SUB_HW * 2, h,
            facecolor=COMP_COLORS[comp], edgecolor="none", zorder=4,
        ))

    ax.plot([veg_left, soil_right], [cy, cy], color="black", linewidth=2.0, zorder=5)

    veg_net = row.get("veg_net", np.nan)
    if not pd.isna(veg_net):
        ax.scatter(cx_veg, cy + veg_net * SCALE, s=30, zorder=6,
                   facecolors="white", edgecolors=VEG_NET_EDGE_COLOR, linewidths=1.5)

    lulucf_net = row.get("total_LULUCF", np.nan)
    if not pd.isna(lulucf_net):
        ax.scatter(cx, cy + lulucf_net * SCALE, s=35, color="black", zorder=6)

    overall_neg_bot = min(veg_neg_bot, soil_neg_bot, 0)
    ax.text(cx, cy + overall_neg_bot - 150_000, region,
            ha="center", va="top", fontsize=7.8, zorder=7)

legend_handles = [
    mpatches.Patch(facecolor=COMP_COLORS["veg_gross_emis"],  label="Vegetation gross emissions"),
    mpatches.Patch(facecolor=COMP_COLORS["veg_gross_remov"], label="Vegetation gross removals"),
    plt.Line2D([0], [0], marker="o", color="w", markersize=7,
               markerfacecolor="white", markeredgecolor=VEG_NET_EDGE_COLOR,
               markeredgewidth=1.5, label="Vegetation net flux"),
    mpatches.Patch(facecolor=COMP_COLORS["organic_soil"],    label="Organic soil emissions"),
    mpatches.Patch(facecolor=COMP_COLORS["mineral_soil"],    label="Mineral soil net flux"),
    plt.Line2D([0], [0], marker="o", color="w", markersize=7,
               markerfacecolor="black", label="LULUCF net flux"),
]
ax.legend(handles=legend_handles,
          bbox_to_anchor=(bounds[0] + 4_200_000, bounds[1] + 300_000),
          bbox_transform=ax.transData, loc="lower left",
          fontsize=9, framealpha=0.9, borderpad=0.7)

# Change 1: scale box moved right of legend (legend at bounds[0]+4_200_000)
sb_x  = bounds[0] + 8_100_000
sb_y  = bounds[1] + 320_000
ref_w = SUB_HW * 1.5
ref_h_500  = 500e6  * SCALE
ref_h_1000 = 1000e6 * SCALE

# Change 2: title above scale box
ax.text(sb_x + ref_w / 2, sb_y + ref_h_1000 + 150_000,
        "Average annual flux, \n2016–2024",
        ha="left", va="bottom", fontsize=7.5, zorder=8)

ax.add_patch(mpatches.Rectangle(
    (sb_x, sb_y), ref_w, ref_h_1000,
    facecolor="#cccccc", edgecolor="#888888", linewidth=0.5, zorder=7,
))
ax.plot([sb_x, sb_x + ref_w], [sb_y + ref_h_500, sb_y + ref_h_500],
        color="#888888", linewidth=0.8, zorder=8)

tick_x = sb_x + ref_w + 100_000
ax.text(tick_x, sb_y,              "0",                  va="center", fontsize=7.5, zorder=8)
ax.text(tick_x, sb_y + ref_h_500,  "500 Mt",             va="center", fontsize=7.5, zorder=8)
ax.text(tick_x, sb_y + ref_h_1000, "1000 Mt CO₂e yr⁻¹", va="center", fontsize=7.5, zorder=8)

# ax.set_title("Average annual LULUCF flux components by region", fontsize=13, pad=10)
plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/regional_map_flux_bar_graphs_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### Average annual fluxes bar chart: gross emissions | gross removals | net flux
### With Claude session "Annual net fluxes bar graph"

# ── Colors ────────────────────────────────────────────────────────────────────
C_EMIS_NONTALL = "#e8b0e8"   # light pink   — short veg & cropland
C_EMIS_DISTURB = "#d279d2"   # medium pink  — partially disturbed
C_EMIS_LOSS    = "#8b008b"   # dark magenta — tree cover loss
C_MIN_SOIL     = "#8c564b"   # brown        — mineral soil
C_ORG_SOIL     = "#993399"   # dark purple  — organic soil (combined)

C_REMOV_UNDIST = "#006600"   # dark green   — undisturbed tree cover
C_REMOV_NEW    = "#33cc33"   # medium green — new tree cover

C_NET          = "#616161"   # gray         — LULUCF net flux

# ── Data aggregation ──────────────────────────────────────────────────────────
_gross = LULUCF_outputs_dropped[
    LULUCF_outputs_dropped["analysis_layer"].str.contains("gross_emissions|gross_removals")
].copy()
_net = LULUCF_outputs_dropped[
    ~LULUCF_outputs_dropped["analysis_layer"].str.contains("gross_emissions|gross_removals")
].copy()

_n_yrs_net = _net["year"].nunique()

_emis_layer  = f"veg_{cn.gross_emis_all_C_pools_all_gases_pattern}"
_remov_layer = f"veg_{cn.gross_removals_all_C_pools_pattern}"

_n_yrs_emis  = _gross.loc[_gross["analysis_layer"] == _emis_layer,  "year"].nunique()
_n_yrs_remov = _gross.loc[_gross["analysis_layer"] == _remov_layer, "year"].nunique()

def _Gt(df, mask, n_yrs):
    return df.loc[mask, "flux_Mg_CO2e_yr"].sum() / n_yrs / 1e9

_e   = _gross["analysis_layer"] == _emis_layer
_r   = _gross["analysis_layer"] == _remov_layer
dlc  = _gross["land_state_detailed_class"]

# Gross emissions
veg_emis_nontall_Gt = _Gt(_gross, _e & (_gross["land_state_broad_class"] != "tree"),                        _n_yrs_emis)
veg_emis_disturb_Gt = _Gt(_gross, _e & dlc.isin(["tree_tree_disturbed", "tree_tree_disturbed_fire_only"]),  _n_yrs_emis)
veg_emis_loss_Gt    = _Gt(_gross, _e & (dlc == "tree_loss"),                                                _n_yrs_emis)
mineral_soil_Gt     = _Gt(_net,   _net["LULUCF_component"] == "mineral_soil",                              _n_yrs_net)
org_soil_Gt         = _Gt(_net,   _net["LULUCF_component"] == "organic_soil",                              _n_yrs_net)

# Gross removals
veg_remov_undist_Gt = _Gt(_gross, _r & (dlc == "tree_tree_undisturbed"), _n_yrs_remov)
veg_remov_new_Gt    = _Gt(_gross, _r & (dlc == "tree_gain"),             _n_yrs_remov)

# Net flux
lulucf_net_Gt = _Gt(_net, _net["analysis_layer"] == "LULUCF_net_flux__MgCO2e", _n_yrs_net)

# QC print
for lbl, val in [
    ("Emis: short veg & cropland",  veg_emis_nontall_Gt),
    ("Emis: partially disturbed",   veg_emis_disturb_Gt),
    ("Emis: tree cover loss",       veg_emis_loss_Gt),
    ("Mineral soil",                mineral_soil_Gt),
    ("Organic soil (combined)",     org_soil_Gt),
    ("Remov: undisturbed",          veg_remov_undist_Gt),
    ("Remov: new tree cover",       veg_remov_new_Gt),
    ("LULUCF net flux",             lulucf_net_Gt),
]:
    print(f"  {lbl:<30s}: {val:+.2f} Gt CO2e/yr")

# ── Bar positions ─────────────────────────────────────────────────────────────
bar_w = 0.6
bsp   = 0.9   # bar spacing within section (center-to-center)

# Left section: gross emissions (5 bars)
x_en  = 0.0              # short veg & cropland
x_ed  = x_en + bsp       # partially disturbed
x_el  = x_ed + bsp       # tree cover loss
x_ms  = x_el + bsp       # mineral soil
x_os  = x_ms + bsp       # organic soil

# Middle section: gross removals (2 bars)
x_ru  = x_os + 1.8       # undisturbed
x_rn  = x_ru + bsp       # new tree cover

# Right section: net flux (1 bar)
x_nf  = x_rn + 1.8

# Section dividers
div1  = (x_os + x_ru) / 2
div2  = (x_rn + x_nf) / 2

# ── Chart ─────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

x_emis, x_remov, x_net = 0.0, 2.5, 4.5
bar_w = 0.7

# Gross emissions — stacked upward
ax.bar(x_emis, veg_emis_nontall_Gt, width=bar_w, color=C_EMIS_NONTALL, zorder=2)
ax.bar(x_emis, veg_emis_disturb_Gt, width=bar_w, color=C_EMIS_DISTURB,
       bottom=veg_emis_nontall_Gt, zorder=2)
ax.bar(x_emis, veg_emis_loss_Gt,    width=bar_w, color=C_EMIS_LOSS,
       bottom=veg_emis_nontall_Gt + veg_emis_disturb_Gt, zorder=2)
ax.bar(x_emis, mineral_soil_Gt,     width=bar_w, color=C_MIN_SOIL,
       bottom=veg_emis_nontall_Gt + veg_emis_disturb_Gt + veg_emis_loss_Gt, zorder=2)
ax.bar(x_emis, org_soil_Gt,         width=bar_w, color=C_ORG_SOIL,
       bottom=veg_emis_nontall_Gt + veg_emis_disturb_Gt + veg_emis_loss_Gt + mineral_soil_Gt, zorder=2)

# Gross removals — stacked downward
ax.bar(x_remov, veg_remov_undist_Gt, width=bar_w, color=C_REMOV_UNDIST, zorder=2)
ax.bar(x_remov, veg_remov_new_Gt,    width=bar_w, color=C_REMOV_NEW,
       bottom=veg_remov_undist_Gt, zorder=2)

# Net flux
ax.bar(x_net, lulucf_net_Gt, width=bar_w, color=C_NET, zorder=2)

# Zero line and section dividers
ax.axhline(0, color="black", linewidth=0.8, zorder=1)
ax.axvline((x_emis + x_remov) / 2, color="black", linewidth=1.2, zorder=1)
ax.axvline((x_remov + x_net)  / 2, color="black", linewidth=1.2, zorder=1)

# Axes formatting
ax.set_xticks([x_emis, x_remov, x_net])
ax.set_xticklabels(["Gross\nemissions", "Gross\nremovals", "LULUCF\nnet flux"])
ax.set_ylabel("Flux (Gt CO$_2$e yr$^{-1}$)")
ax.set_title(
    f"Average annual LULUCF GHG fluxes, "
    f"{cn.interval_end_years_annual[0]}–{cn.interval_end_years_annual[-1]}",
    pad=10,
)
ax.set_xlim(x_emis - bar_w, x_net + bar_w)
for spine in ["top", "right", "bottom"]:
    ax.spines[spine].set_visible(False)
ax.tick_params(axis="x", length=0, labelsize=12)

# Legend — right of figure
legend_handles = [
    mpatches.Patch(color=C_ORG_SOIL,     label="Organic soil"),
    mpatches.Patch(color=C_MIN_SOIL,     label="Mineral soil"),
    mpatches.Patch(color=C_EMIS_LOSS,    label="Tree cover loss"),
    mpatches.Patch(color=C_EMIS_DISTURB, label="Partially disturbed"),
    mpatches.Patch(color=C_EMIS_NONTALL, label="Short veg & cropland"),
    mpatches.Patch(color=C_REMOV_UNDIST, label="Undisturbed tree cover"),
    mpatches.Patch(color=C_REMOV_NEW,    label="New tree cover"),
    mpatches.Patch(color=C_NET,          label="LULUCF net flux"),
]
ax.legend(handles=legend_handles, loc="center left", bbox_to_anchor=(1.02, 0.5),
          frameon=False, fontsize=9)

plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/average_annual_bar_graphs_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
### Global Carbon Budget comparison bar graphs.
### Uses an Excel spreadsheet in which I've already aggregated the GCB values and calculated the LCL values (i.e. translated the LCL values to GCB).
### So, the actual workflow for this is: create LULUCF pandas dataframe from flox zonal stats runs -> ingest into Postgres table -> query postgres table and put results in Excel -> perform GCB translation -> run this script
### I decided it was easier to calculate translated LULUCF framework values and write notes about GCB and LCL data in Excel than doing all of it in a Jupyter notebook. Hence, this route that goes through Excel. 
### With Claude session 'LULUCF vs Global Carbon Budget graphs'

EXCEL_PATH = (
    '/mnt/c/Users/David.Gibbs/OneDrive - World Resources Institute/Documents/Projects'
    '/AFOLU_flux_model__all_land_all_carbon/LULUCF/Global_Carbon_Budget_comparison'
    '/LCL_LULUCF_vs_GCB_land_20260523.xlsx'
)
GCB_COLOR = '#808080'
LCL_COLOR = '#D4A017'

raw = pd.read_excel(EXCEL_PATH, sheet_name='comparison', header=0)
raw = raw.iloc[:, [1, 4, 5, 12, 13, 16]].copy()
raw.columns = ['gcb_label', 'gcb_flux', 'gcb_stdev', 'lcl_label', 'lcl_flux', 'lcl_stdev']

def ix(excel_rows):
    return [r - 2 for r in excel_rows]

def wrap_label(s, width=16):
    return '\n'.join(textwrap.wrap(str(s), width=width))

def parse_stdev(val, central=None):
    if pd.isna(val):
        return np.nan, np.nan
    s = str(val).strip()
    if s.upper() == 'N/A':
        return np.nan, np.nan
    m = re.match(r'\[\s*([+-]?\d*\.?\d+)\s*,\s*([+-]?\d*\.?\d+)\s*\]', s)
    if m:
        lo, hi = float(m.group(1)), float(m.group(2))
        if central is not None:
            return abs(float(central) - lo), abs(hi - float(central))
        return np.nan, np.nan
    try:
        v = float(s)
        return abs(v), abs(v)
    except ValueError:
        return np.nan, np.nan

def add_errorbars(ax, x, y, lo, hi):
    if not (np.isnan(lo) or np.isnan(hi)):
        ax.errorbar(x, y, yerr=[[lo], [hi]], fmt='none',
                    ecolor='black', capsize=4, elinewidth=1.5)

legend_kw = dict(loc='lower left', fontsize=6, handlelength=1.0,
                 handleheight=0.6, borderpad=0.4, labelspacing=0.2)
legend_handles = [
    mpatches.Patch(color=GCB_COLOR, label='GCB'),
    mpatches.Patch(color=LCL_COLOR, label='LCL LULUCF'),
]

# ── Data prep ─────────────────────────────────────────────────────────────────
gcb_L = raw.iloc[ix([2, 3, 4, 5, 7, 8, 9])].reset_index(drop=True)
lcl_L = raw.iloc[ix([8, 9])].reset_index(drop=True)

bars_L = []
for i in range(6):
    bars_L.append({'label': gcb_L.iloc[i]['gcb_label'], 'flux': gcb_L.iloc[i]['gcb_flux'],
                   'stdev': gcb_L.iloc[i]['gcb_stdev'], 'color': GCB_COLOR})
bars_L.append({'label': lcl_L.iloc[0]['lcl_label'], 'flux': lcl_L.iloc[0]['lcl_flux'],
               'stdev': lcl_L.iloc[0]['lcl_stdev'], 'color': LCL_COLOR})
no_organic_start = len(bars_L)   # = 7
bars_L.append({'label': gcb_L.iloc[6]['gcb_label'], 'flux': gcb_L.iloc[6]['gcb_flux'],
               'stdev': gcb_L.iloc[6]['gcb_stdev'], 'color': GCB_COLOR})
bars_L.append({'label': lcl_L.iloc[1]['lcl_label'], 'flux': lcl_L.iloc[1]['lcl_flux'],
               'stdev': lcl_L.iloc[1]['lcl_stdev'], 'color': LCL_COLOR})

mid_included  = (no_organic_start - 1) / 2        # centre of positions 0–6  = 3.0
mid_excluded  = (no_organic_start + len(bars_L) - 1) / 2   # centre of 7–8 = 7.5

both_R = raw.iloc[ix([11, 14, 15, 16, 17, 18, 19, 20, 21, 22, 25])].reset_index(drop=True)
x_c = np.arange(len(both_R))
bw  = 0.4

# ── Panel draw functions ──────────────────────────────────────────────────────
def draw_left(ax):
    ax.yaxis.grid(True, linewidth=0.5, color='lightgrey')
    ax.set_axisbelow(True)
    for i, bar in enumerate(bars_L):
        lo, hi = parse_stdev(bar['stdev'], bar['flux'])
        ax.bar(i, bar['flux'], color=bar['color'])
        add_errorbars(ax, i, bar['flux'], lo, hi)

    # Section divider and labels (xaxis transform: data-x, axes-fraction-y)
    ax.axvline(x=no_organic_start - 0.5, color='gray', linewidth=1.0)
    ax.text(mid_included, 0.97, 'Organic soil included',
            ha='center', va='top', fontsize=8, color='dimgray', style='italic',
            transform=ax.get_xaxis_transform())
    ax.text(mid_excluded, 0.97, 'Organic soil excluded',
            ha='center', va='top', fontsize=8, color='dimgray', style='italic',
            transform=ax.get_xaxis_transform())

    ax.set_xticks(np.arange(len(bars_L)))
    ax.set_xticklabels([wrap_label(bar['label']) for bar in bars_L],
                       rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gt CO₂ yr⁻¹')
    ax.set_title('Net atmosphere-land flux comparison\n(GCB: 2015–2024; LCL 2016–2024)')
    ax.legend(handles=legend_handles, **legend_kw)
    yabs = max(abs(ax.get_ylim()[0]), abs(ax.get_ylim()[1]))
    ax.set_ylim(-yabs, yabs)

def draw_right(ax):
    ax.yaxis.grid(True, linewidth=0.5, color='lightgrey')
    ax.set_axisbelow(True)
    ax.bar(x_c - bw / 2, both_R['gcb_flux'], width=bw, color=GCB_COLOR)
    for i, row in both_R.iterrows():
        lo, hi = parse_stdev(row['gcb_stdev'], row['gcb_flux'])
        add_errorbars(ax, x_c[i] - bw / 2, row['gcb_flux'], lo, hi)
    ax.bar(x_c + bw / 2, both_R['lcl_flux'], width=bw, color=LCL_COLOR)
    for i, row in both_R.iterrows():
        lo, hi = parse_stdev(row['lcl_stdev'], row['lcl_flux'])
        add_errorbars(ax, x_c[i] + bw / 2, row['lcl_flux'], lo, hi)

    # Section divider and labels
    s_land_x = x_c[-1]                      # position of S_land bar group = 10
    mid_eluc  = (x_c[0] + x_c[-2]) / 2     # centre of E_luc bars (0–9) = 4.5
    ax.axvline(x=s_land_x - 0.5, color='gray', linewidth=1.0)
    ax.text(mid_eluc, 0.97, 'E_luc',
            ha='center', va='top', fontsize=8, color='dimgray', style='italic',
            transform=ax.get_xaxis_transform())
    ax.text(s_land_x, 0.97, 'S_land',
            ha='center', va='top', fontsize=8, color='dimgray', style='italic',
            transform=ax.get_xaxis_transform())

    ax.set_xticks(x_c)
    ax.set_xticklabels([wrap_label(l) for l in both_R['gcb_label']],
                       rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Gt CO₂ yr⁻¹')
    ax.set_title('E_luc and S_land comparison\n(GCB: 2015–2024; LCL 2016–2024)')
    ax.legend(handles=legend_handles, **legend_kw)
    yabs = max(abs(ax.get_ylim()[0]), abs(ax.get_ylim()[1]))
    ax.set_ylim(-yabs, yabs)

# ── Individual figure a ───────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(12, 6))
draw_left(ax1)
plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_GCB_land_atmos_flux_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# ── Individual figure b ───────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(14, 6))
draw_right(ax2)
plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_GCB_Eluc_Sland_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# ── Combined figure (a above b) ───────────────────────────────────────────────
fig_c, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(14, 12))
draw_left(ax_a)
draw_right(ax_b)
for ax, label in zip([ax_a, ax_b], ['a', 'b']):
    ax.text(0.01, 0.98, label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='left')
plt.tight_layout()
# plt.savefig(f'{LULUCF_zonal_stats_folder}/LULUCF_GCB_two_panel_comparison_{today}.jpg', dpi=300, bbox_inches='tight')
plt.show()

# ── Print values ──────────────────────────────────────────────────────────────
def fmt_err(val, central=None):
    lo, hi = parse_stdev(val, central)
    if np.isnan(lo):
        return 'N/A'
    return f'±{lo:.2f}' if lo == hi else f'+{hi:.2f}/-{lo:.2f}'

print('=== Panel a: Net atmosphere-land flux ===')
print(f'\n{"Label":<45} {"Source":>6} {"Flux":>10} {"Uncertainty":>20}')
print('-' * 83)
for bar in bars_L:
    src = 'GCB' if bar['color'] == GCB_COLOR else 'LCL'
    print(f'{str(bar["label"]):<45} {src:>6} {bar["flux"]:>10.2f} {fmt_err(bar["stdev"], bar["flux"]):>20}')

print('\n=== Panel b: E_luc and S_land ===')
print(f'\n{"Label":<45} {"GCB flux":>10} {"GCB uncert":>15} {"LCL flux":>10} {"LCL uncert":>15}')
print('-' * 97)
for _, row in both_R.iterrows():
    print(f'{str(row["gcb_label"]):<45} {row["gcb_flux"]:>10.2f} '
          f'{fmt_err(row["gcb_stdev"], row["gcb_flux"]):>15} '
          f'{row["lcl_flux"]:>10.2f} {fmt_err(row["lcl_stdev"], row["lcl_flux"]):>15}')